# Reproducing Boran et al. (2019) from dandiset 000574

Paper: Boran, Fedele, Klaver, Hilfiker, Stieglitz, Grunwald & Sarnthein (2019), "Persistent
hippocampal neural firing and hippocampal-cortical coupling predict verbal working memory
load", Science Advances 5, eaav3687, doi:10.1126/sciadv.aav3687.

Dataset: [DANDI:000574](https://dandiarchive.org/dandiset/000574/0.250815.1108), version
0.250815.1108 (doi:10.48324/dandi.000574/0.250815.1108). The paper cites no version of the
dandiset (it predates it and points to the authors on request), so the most recent published
version is pinned. Version 0.250610.0938 holds the same 37 assets (identical asset ids). The
earlier version 0.231010.1809 held 45 files, with sessions of subjects 2, 5, 7 and 9 deposited
more than once under `obj-` names; it matches the claims file's description no better.

Claims file: `claims/000574/10.1126-sciadv.aav3687.yaml`, written from the version of record
(Europe PMC, PMC6436923) with its supplement, and the data descriptor (Boran et al. 2020,
Sci Data 7, 30).

Every number below is computed from the streamed NWB files. The table of claims and outcomes
is in the summary at the end.

## Claims

| Claim | Tested |
|---|---|
| behavior-accuracy-set-size | yes |
| behavior-capacity | yes |
| behavior-rt-workload | yes |
| behavior-rt-correct-incorrect | yes |
| neurons-recorded | yes |
| spike-sorting-quality | yes |
| fig2b-maintenance-neurons-hippocampus | yes |
| fig2c-load-dependent-maintenance-neurons | yes |
| fig3e-probe-neurons-entorhinal | yes |
| fig3-probe-onset-not-movement | yes |
| fig3-probe-neurons-few-task-specific | yes |
| fig4a-dpca-workload-separation | yes |
| fig4b-speed-lowest-maintenance | yes |
| fig4c-distance-largest-maintenance | yes |
| fig4d-encoding-decodes-correct | yes |
| fig4e-maintenance-decodes-workload | yes |
| fig4e-workload-decoding-leave-one-subject-out | yes |
| fig4e-maintenance-neurons-carry-workload | yes |
| fig4e-no-decoding-6-vs-8 | yes |
| fig4f-retrieval-decodes-response | yes |
| figs3d-hippocampus-loadings | yes |
| fig5-subject1-eeg-power-workload | yes |
| fig5-subject1-hippocampal-power-workload | yes |
| fig5-subject1-neurons-workload | yes |
| fig5-subject1-plv-hippocampus-pz | yes |
| fig6-plv-all-subjects | yes |
| tables1-plv-per-subject | yes |
| fig6-plv-workload-eight-of-nine | yes |
| fig6-plv-maintenance-over-encoding | yes |
| fig6-power-unchanged-individual-band | yes |
| fig6i-plv-posterior-gradient | yes |
| fig6i-no-left-right-difference | yes |
| soz-plv-no-difference | yes |
| hfo-rates-unaffected | no, blocked |
| sfc-maintenance-over-encoding | yes, in reproduce_boran_2019_sfc.ipynb |

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import sternberg as S
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)

In [ ]:
A = S.assets()
D = S.load_all(A, workers=16)
print(len(A), "assets read at version", S.DANDISET_VERSION)

## Data check

Counts by subject and session, trial fields, units and their regions, and the checks of the
description against the values.

In [ ]:
T = pd.concat([d["trials"].assign(subject=d["subject"], session=d["session"]) for d in D],
              ignore_index=True)
sess = pd.DataFrame([{"subject": d["subject"], "session": d["session"],
                      "start": d["meta"]["start_time"], "trials": len(d["trials"]),
                      "correct": int(d["trials"].correct.sum()),
                      "artifact": int(d["trials"].artifact.sum()), "units": len(d["units"]),
                      "eeg_ch": d["shapes"]["eeg"][1], "ieeg_ch": d["shapes"]["ieeg"][1],
                      "eeg_rate": d["shapes"]["eeg"][0] / (8 * len(d["trials"])),
                      "ieeg_rate": d["shapes"]["ieeg"][0] / (8 * len(d["trials"])),
                      "lfp_rate": d["shapes"]["lfp"][0] / (8 * len(d["trials"]))} for d in D])
print(sess.to_string())
print("\nsubjects", sess.subject.nunique(), "sessions", len(sess), "per subject",
      sess.groupby("subject").size().to_dict())
print("trials", len(T), "correct", int(T.correct.sum()), "artifact-marked", int(T.artifact.sum()))
print("sessions with fewer than 50 trials:", int((sess.trials < 50).sum()),
      "; sessions without a units table:", int((sess.units == 0).sum()))
print("trial duration (s):", np.unique(np.round(T.stop - T.start, 6)))
print("set sizes:", T.set_size.value_counts().sort_index().to_dict())
print("response codes by correctness and solution:")
print(pd.crosstab([T.correct, T.solution], T.response))
print("RT relative to probe onset (s): min %.3f median %.3f max %.3f" % (T.rt.min(), T.rt.median(), T.rt.max()))

In [ ]:
# The trial-stage table: its Probe stages are numbered one trial late, and some files lack it.
chk = []
for d in D:
    st, tr = d["stages"], d["trials"]
    if len(st) == 0:
        chk.append((d["subject"], d["session"], "no ttl_stage", np.nan, np.nan))
        continue
    pr = st[st.tag == "Probe"].sort_values("start").start.to_numpy()
    k = min(len(pr), len(tr))
    off = np.abs(pr[:k] - tr.probe.to_numpy()[:k]).max()
    fx = st[st.tag == "Fixation"].sort_values("start").start.to_numpy()
    offf = np.abs(fx[:len(tr)] - tr.start.to_numpy()[:len(fx)]).max()
    chk.append((d["subject"], d["session"], "ok", off, offf))
chk = pd.DataFrame(chk, columns=["subject", "session", "stages", "max |probe stage - (start+6)|",
                                 "max |fixation stage - start|"])
print(chk.to_string())

In [ ]:
# Units: regions from the bundle name (claims file) and from the atlas label of the bundle's
# deepest contact (Methods, Depth electrode localization).
rows = []
for d in D:
    lab = dict(zip(d["electrodes"].label, d["electrodes"].location))
    for u in d["units"]:
        loc1 = lab.get(u["bundle"] + "1", "?").split(",")[0]
        atlas = {"Hipp": "hippocampus", "PhG": "entorhinal", "Amyg": "amygdala"}.get(loc1, "other:" + loc1)
        sp = u["spikes"]
        rows.append({"subject": d["subject"], "session": d["session"], "unit": u["unit"],
                     "bundle": u["bundle"], "region_bundle": u["region"], "atlas_label": loc1,
                     "region_atlas": atlas, "n_spikes": len(sp),
                     "sorted": u["sorted_in_file"],
                     "in_obs": float(np.mean([np.any((s >= u["obs"][:, 0]) & (s <= u["obs"][:, 1])) for s in sp[:200]])) if len(sp) else np.nan})
U = pd.DataFrame(rows)
print("units", len(U), "; units whose spike times are stored unsorted:", int((~U.sorted).sum()),
      U[~U.sorted].groupby(["subject", "session"]).size().to_dict(),
      "; fraction of (first 200) spikes inside the unit's obs_intervals:", U.in_obs.min())
print(pd.crosstab(U.region_bundle, U.region_atlas, margins=True))
print(pd.crosstab([U.subject, U.bundle], U.atlas_label))

In [ ]:
# Electrode coordinates: the table says +x is posterior, +y inferior, +z right. Check on the
# bundles: posterior hippocampus (PH) should lie at larger x than anterior (AH), right
# electrodes at positive z, and contact 1 should be the deepest (nearest the midline, smallest |z|).
ec = pd.concat([d["electrodes"].assign(subject=d["subject"]) for d in D if d["session"] == "01"])
ie = ec[ec.group_name == "ieeg"].copy()
ie["bundle"] = ie.label.str.replace(r"\d+$", "", regex=True)
ie["contact"] = ie.label.str.extract(r"(\d+)$").astype(float)
mx = ie.groupby(["subject", "bundle"]).apply(lambda g: pd.Series({
    "x1": g[g.contact == 1].x.mean(), "z1": g[g.contact == 1].z.mean(),
    "absz_1_lt_8": float(abs(g[g.contact == 1].z.mean()) < abs(g[g.contact == g.contact.max()].z.mean()))}))
mx = mx.reset_index()
ah = mx[mx.bundle.str.startswith("AH")].groupby("subject").x1.mean()
ph = mx[mx.bundle.str.startswith("PH")].groupby("subject").x1.mean()
print("x of contact 1, AH and PH per subject:\n", pd.concat([ah, ph], axis=1, keys=["AH", "PH"]))
print("right bundles with z > 0:", float((mx[mx.bundle.str.endswith("R")].z1 > 0).mean()),
      "; left bundles with z < 0:", float((mx[mx.bundle.str.endswith("L")].z1 < 0).mean()))
print("bundles whose contact 1 is nearer the midline than their last contact:", mx.absz_1_lt_8.mean())
print("SOZ electrodes per subject (is_inside_soz) and the SOZ list in the subject description:")
for d in D:
    if d["session"] != "01":
        continue
    e = d["electrodes"]
    soz = sorted(set(e[e.is_inside_soz].label.str.replace(r"\d+$", "", regex=True)))
    desc = [l for l in d["meta"]["description"].split("\n") if "SOZ" in l][0]
    print(" ", d["subject"], soz, "|", desc)

## Behavior

All trials of all 37 sessions (the paper has 36; the dataset holds 37, as the data
descriptor's Table 1 lists). 13 sessions hold fewer than 50 trials.

### behavior-accuracy-set-size

> Correct response rates were 90.2 +/- 7.8% for IN and 93.1 +/- 4.3% for OUT trials, and the
> rate of correct responses decreased with set size, from 98.5% (set size 4) to 90.5% (6) and
> 84.7% (8), a significant effect in a permuted repeated-measures ANOVA over 36 sessions.

IN and OUT trials are those whose probe was or was not in the set (`solution`), as Cowan's K
uses "correct IN rate" for hits.

In [ ]:
rng = np.random.default_rng(101)
T["sess"] = T.subject + "-" + T.session
per = T.groupby("sess").apply(lambda x: pd.Series({
    "IN": 100 * x[x.solution].correct.mean(), "OUT": 100 * x[~x.solution].correct.mean(),
    **{f"ss{k}": 100 * x[x.set_size == k].correct.mean() for k in (4, 6, 8)}}))
pooled = {"IN": 100 * T[T.solution].correct.mean(), "OUT": 100 * T[~T.solution].correct.mean(),
          **{f"ss{k}": 100 * T[T.set_size == k].correct.mean() for k in (4, 6, 8)}}
print("(a) mean of per-session percentages:", per.mean().round(2).to_dict())
print("    SD over sessions:", per.std().round(2).to_dict())
print("(b) trials pooled:", {k: round(v, 2) for k, v in pooled.items()})


def rm_anova_F(Y):
    """One-way repeated-measures F; Y is sessions x conditions."""
    n, k = Y.shape
    gm = Y.mean()
    ss_c = n * ((Y.mean(0) - gm) ** 2).sum()
    ss_s = k * ((Y.mean(1) - gm) ** 2).sum()
    ss_e = ((Y - gm) ** 2).sum() - ss_c - ss_s
    return (ss_c / (k - 1)) / (ss_e / ((n - 1) * (k - 1))), (k - 1, (n - 1) * (k - 1))


Y = per[["ss4", "ss6", "ss8"]].to_numpy()
F0, dfs = rm_anova_F(Y)
nperm = 4999
Fn = np.array([rm_anova_F(np.array([rng.permutation(r) for r in Y]))[0] for _ in range(nperm)])
p_anova = (np.sum(Fn >= F0) + 1) / (nperm + 1)
print(f"repeated-measures ANOVA over {Y.shape[0]} sessions: F{dfs} = {F0:.2f}, permutation P = {p_anova:.4f}"
      f" (labels shuffled within session, {nperm} permutations)")
# Same without the one session the paper may not have had: leave each session out in turn
Fl = [rm_anova_F(np.delete(Y, i, 0))[0] for i in range(len(Y))]
print("F with one session left out, range: %.2f to %.2f" % (min(Fl), max(Fl)))
print("per subject (mean of session percentages):")
print(per.groupby(per.index.str[:2]).mean().round(1))

**Result: Reproduced.** Over the 37 sessions the mean of per-session percentages is 90.26% for IN and 93.12% for OUT trials, and 98.44%, 90.47% and 85.12% for set sizes 4, 6 and 8, all within tolerance; pooling trials gives the same within 0.4 points. The repeated-measures ANOVA gives F(2, 72) = 37.28 (paper 40.25 +/- 4.03), permutation P = 0.0002. The data hold 37 sessions where the paper has 36.

### behavior-capacity

> Across all sessions the working memory capacity (Cowan's K) averaged 6.8 letters.

In [ ]:
def cowan(x, k):
    return (x[x.solution].correct.mean() + x[~x.solution].correct.mean() - 1) * k


K = T.groupby("sess").apply(lambda x: pd.Series({
    "a_K8": cowan(x[x.set_size == 8], 8),
    "b_maxK": max(cowan(x[x.set_size == k], k) for k in (4, 6, 8)),
    "c_pooled_x8": cowan(x, 8),
    "d_meanK": np.mean([cowan(x[x.set_size == k], k) for k in (4, 6, 8)])}))
print("mean Cowan's K over sessions, by reading:")
print(K.mean().round(2).to_string())
print("range of K over sessions:", K.min().round(2).to_dict(), K.max().round(2).to_dict())

**Result: Reproduced under reading (c), rates pooled over set sizes times 8: mean K = 6.**67 (paper 6.8 +/- 0.68). Readings (a) 5.53, (b) 6.01 and (d) 4.71 are outside the tolerance.

### behavior-rt-workload

> The mean response time of correct trials (1630 trials) increased with workload, by 48 ms per
> item, a significant effect in a permuted repeated-measures ANOVA.

In [ ]:
rng = np.random.default_rng(102)
C = T[T.correct].copy()
print("correct trials:", len(C), "; mean RT by set size (s):", C.groupby("set_size").rt.mean().round(4).to_dict())


def oneway_F(y, g):
    groups = [y[g == k] for k in np.unique(g)]
    return stats.f_oneway(*groups).statistic


F0 = oneway_F(C.rt.to_numpy(), C.set_size.to_numpy())
Fn = np.array([oneway_F(C.rt.to_numpy(), rng.permutation(C.set_size.to_numpy())) for _ in range(4999)])
print(f"one-way ANOVA over correct trials: F(2, {len(C) - 3}) = {F0:.2f}, permutation P = {(np.sum(Fn >= F0) + 1) / 5000:.4f}")
slope_i = np.polyfit(C.set_size, C.rt, 1)[0] * 1000
ps_mean = C.groupby("sess").apply(lambda x: np.polyfit(x.set_size, x.rt, 1)[0] * 1000 if x.set_size.nunique() > 1 else np.nan)
med = C.groupby(["sess", "set_size"]).rt.median().unstack()
ps_med = med.apply(lambda r: np.polyfit([4, 6, 8], r.to_numpy(), 1)[0] * 1000 if r.notna().all() else np.nan, axis=1)
m3 = C.groupby("set_size").rt.mean()
slope_iii = np.polyfit(m3.index, m3.to_numpy(), 1)[0] * 1000
slope_iv = (m3[8] - m3[4]) / 4 * 1000
print("RT increase per item (ms): (i) line over trials %.1f; (ii) mean of per-session slopes, trial RTs %.1f,"
      " session medians %.1f; (iii) slope of set-size means %.1f; (iv) (8 - 4) / 4 %.1f"
      % (slope_i, ps_mean.mean(), ps_med.mean(), slope_iii, slope_iv))
print("per subject slope (i):", C.groupby("subject").apply(lambda x: round(np.polyfit(x.set_size, x.rt, 1)[0] * 1000, 1)).to_dict())

**Result: Reproduced.** 1676 correct trials (paper 1630 +/- 163); RT rises from 1.262 s (set size 4) to 1.420 s (6) and 1.443 s (8); the slope over trials is 47.0 ms per item (paper 48 +/- 4.8), and per-session slopes give 48.7 ms; the one-way ANOVA over trials has F(2, 1673) = 14.98, permutation P = 0.0002 (the paper's F of 21.10 on 2 and 2007 degrees of freedom is reported, not judged).

### behavior-rt-correct-incorrect

> Correct IN/OUT decisions were made faster than incorrect ones (1.36 +/- 0.64 s against
> 1.87 +/- 1.21 s).

In [ ]:
rng = np.random.default_rng(103)
rc, ri = T[T.correct].rt.to_numpy(), T[~T.correct].rt.to_numpy()
t_a, p_a = S.perm_t(ri, rc, n_perm=2000, rng=rng)
print(f"(a) pooled trials: correct {rc.mean():.3f} +/- {rc.std(ddof=1):.3f} s (n = {len(rc)}), incorrect "
      f"{ri.mean():.3f} +/- {ri.std(ddof=1):.3f} s (n = {len(ri)}); unpaired permutation t = {t_a:.2f}, P = {p_a:.4f}")
sm = T.groupby(["sess", "correct"]).rt.mean().unstack().dropna()
t_b, p_b = S.perm_t(sm[False], sm[True], n_perm=2000, paired=True, rng=rng)
print(f"(b) per-session means, {len(sm)} sessions with an error: correct {sm[True].mean():.3f} +/- {sm[True].std():.3f},"
      f" incorrect {sm[False].mean():.3f} +/- {sm[False].std():.3f}; paired permutation t = {t_b:.2f}, P = {p_b:.4f}")
print("medians (s): correct %.3f, incorrect %.3f" % (np.median(rc), np.median(ri)))

**Result: Reproduced.** Pooled trials: correct 1.361 +/- 0.632 s, incorrect 1.894 +/- 1.228 s (paper 1.36 and 1.87), unpaired permutation t = 5.27, P = 0.0005; per-session means give the same direction (paired t = 5.73, P = 0.0005).

## Neurons

Spike times are aligned to each trial's probe onset (trial start + 6 s) and counted in 20 ms
bins from -6 to +2 s, the 8 s that each trial stores. Sliding windows (500 ms or 200 ms at
20 ms steps, Fig. 2A and Fig. 3 legends) are sums of these bins.

Regions. The claims file defines them for `neurons-recorded` by the bundle name (AH and PH
hippocampus, EC entorhinal cortex, A amygdala). The other claims do not say; the paper assigns
electrodes to regions with the Brainnetome atlas (Methods, Depth electrode localization: "The
contact positions were normalized to the MNI space and assigned to a brain region using the
Brainnetome Atlas"), and the microwires protrude from the tip, next to contact 1. So for those
claims the main reading is the atlas label of the bundle's contact 1 (Hipp hippocampus, PhG
entorhinal cortex, Amyg amygdala; 28 units under FuG are left out), and the bundle reading is
shown as a variant.

In [ ]:
EDGES = np.round(np.arange(-6.0, 2.0 + 1e-9, 0.02), 4)
units = []
for d in D:
    tr = d["trials"]
    lab = dict(zip(d["electrodes"].label, d["electrodes"].location))
    for u in d["units"]:
        al = S.aligned_spikes(u["spikes"], tr.probe.to_numpy())
        loc1 = lab.get(u["bundle"] + "1", "?").split(",")[0]
        units.append({"subject": d["subject"], "session": d["session"], "sess": d["subject"] + "-" + d["session"],
                      "unit": u["unit"], "bundle": u["bundle"], "region_bundle": u["region"],
                      "region_atlas": {"Hipp": "hippocampus", "PhG": "entorhinal", "Amyg": "amygdala"}.get(loc1, "other"),
                      "aligned": al, "bins": S.binned(al, EDGES).astype(np.int16),
                      "correct": tr.correct.to_numpy(), "set_size": tr.set_size.to_numpy(),
                      "is_in": tr.is_in.to_numpy(), "rt": tr.rt.to_numpy(),
                      "spikes": u["spikes"], "obs": u["obs"],
                      "wm": u["waveform_mean"], "wsd": u["waveform_sd"]})
UN = pd.DataFrame([{k: v for k, v in u.items() if k in ("subject", "session", "sess", "unit", "bundle", "region_bundle", "region_atlas")} for u in units])
REG = ["hippocampus", "entorhinal", "amygdala"]
print(len(units), "units;", UN.region_atlas.value_counts().to_dict())


def win_counts(u, window):
    i0 = int(round((window[0] + 6) / 0.02))
    i1 = int(round((window[1] + 6) / 0.02))
    return u["bins"][:, i0:i1].sum(1).astype(float)


def win_rate(u, window):
    return win_counts(u, window) / (window[1] - window[0])


def sliding(u, width, lo, hi, step=0.02):
    """Rate in windows of `width` s centred on times lo..hi at 20 ms steps (trials x times)."""
    w = int(round(width / 0.02))
    cs = np.concatenate([np.zeros((u["bins"].shape[0], 1)), np.cumsum(u["bins"], 1)], 1)
    centres = np.round(np.arange(lo, hi + 1e-9, step), 4)
    out = []
    for c in centres:
        a = int(round((c - width / 2 + 6) / 0.02))
        a = min(max(a, 0), u["bins"].shape[1] - w)
        out.append((cs[:, a + w] - cs[:, a]) / width)
    return np.array(out).T, centres

### neurons-recorded

> 1526 neurons were recorded (51 +/- 35 per session), 676 in hippocampus, 449 in entorhinal
> cortex and 401 in amygdala.

In [ ]:
cnt_b = UN.region_bundle.value_counts()
cnt_a = UN.region_atlas.value_counts()
print("units:", len(UN))
print("by bundle name (claims file):", cnt_b.to_dict())
print("by atlas label of contact 1 (variant):", cnt_a.to_dict())
per_sess = sess.set_index(sess.subject + "-" + sess.session).units
print("per session, (a) all 37 sessions: %.1f +/- %.1f; (b) the %d sessions with units: %.1f +/- %.1f"
      % (per_sess.mean(), per_sess.std(), (per_sess > 0).sum(), per_sess[per_sess > 0].mean(), per_sess[per_sess > 0].std()))
print("sessions without units:", list(per_sess[per_sess == 0].index))
print("units per subject:", UN.groupby("subject").size().to_dict())
print("differences from the paper (bundle): total %+d, hippocampus %+d, entorhinal %+d, amygdala %+d"
      % (len(UN) - 1526, cnt_b["hippocampus"] - 676, cnt_b["entorhinal"] - 449, cnt_b["amygdala"] - 401))

**Result: Not reproduced.** The dataset holds 1487 units (paper 1526). By bundle name, as the criterion defines the regions, 446 are hippocampal (paper 676), 456 entorhinal (449) and 585 amygdalar (401). By the atlas label of each bundle's deepest contact the counts are 640, 421 and 398 (28 under fusiform gyrus), closer to the paper. Per session, 51.3 +/- 35.6 over the 29 sessions with units matches the paper's 51 +/- 35; 8 of the 37 sessions hold no units.

### spike-sorting-quality

> The mean firing rate of all neurons was 2.55 +/- 3.30 Hz; the percentage of interspike
> intervals below 3 ms was 1.58 +/- 4.18%, and the ratio of the peak of the mean waveform to
> the SD of the noise was 3.28 +/- 1.56.

In [ ]:
q = []
for u in units:
    obs = u["obs"]
    dur_trials = float(np.sum(obs[:, 1] - obs[:, 0]))
    dur_session = float(obs[:, 1].max() - obs[:, 0].min())
    sp = u["spikes"]
    isi = np.diff(sp)
    # ISIs within one stored trial only (the gaps between trials are not recorded)
    same = np.searchsorted(obs[:, 0], sp[1:], side="right") == np.searchsorted(obs[:, 0], sp[:-1], side="right")
    isi = isi[same]
    peak = np.max(np.abs(u["wm"]))
    q.append({"rate_trials": len(sp) / dur_trials, "rate_session": len(sp) / dur_session,
              "isi3": 100 * np.mean(isi < 0.003) if len(isi) else np.nan,
              "snr_meansd": peak / np.mean(u["wsd"]), "snr_first": peak / np.mean(u["wsd"][:5])})
Q = pd.DataFrame(q)
print(Q.agg(["mean", "std", "median", "min", "max"]).round(3).to_string())
print("units with ISI computable:", int(Q.isi3.notna().sum()), "of", len(Q))
print("waveform: %d samples (2 ms at 32 kHz); peak |mean| range %.1f to %.1f uV" %
      (len(units[0]["wm"]), 1e6 * min(np.abs(u["wm"]).max() for u in units), 1e6 * max(np.abs(u["wm"]).max() for u in units)))

**Result: Partially reproduced.** The mean firing rate over the stored trials is 2.57 +/- 3.33 Hz (paper 2.55 +/- 3.30) and the percentage of ISIs below 3 ms is 1.75 +/- 4.59% (paper 1.58 +/- 4.18), both within tolerance; the waveform SNR is 5.34 under both noise readings (paper 3.28 +/- 1.56), outside it. The noise SD of the paper is not in the files; the deposited waveform SD is a different quantity.

### fig2b-maintenance-neurons-hippocampus

> Neurons with higher firing during maintenance than fixation (maintenance neurons) occurred in
> all three areas, most frequently in hippocampus, where 20.6% of the recorded neurons were
> maintenance neurons.

Correct trials; rate in maintenance (-3 to 0 s) against fixation (-6 to -5 s), permutation t
test with 2000 permutations, P < 0.05 and maintenance higher; paired and unpaired readings.
The null shuffles the maintenance and fixation labels within each trial and reruns the
selection, 1000 times (P also given from the first 499 repetitions).

In [ ]:
rng = np.random.default_rng(201)
R = 1000
sel = {"paired": [], "unpaired": []}
null_sel = {"paired": [], "unpaired": []}
p_mf = {"paired": [], "unpaired": []}
for u in units:
    c = u["correct"]
    m, f = win_rate(u, S.MAINT)[c], win_rate(u, S.FIX)[c]
    n = len(m)
    d = m - f
    # paired: sign flips
    t_ref = S.t_paired_rows(rng.choice([-1.0, 1.0], (2000, n)) * d)
    t0 = S.t_paired_rows(d[None])[0]
    p0 = S.perm_p(t0, t_ref)
    tr_ = S.t_paired_rows(rng.choice([-1.0, 1.0], (R, n)) * d)
    sel["paired"].append(bool(p0 < 0.05 and t0 > 0))
    null_sel["paired"].append((S.perm_p(tr_, t_ref) < 0.05) & (tr_ > 0))
    p_mf["paired"].append(p0)
    # unpaired: values of the two periods pooled
    x = np.concatenate([m, f])[:, None]
    t_ref = S.t_groups(x, S.random_masks(n, 2 * n, 2000, rng))[:, 0]
    g0 = np.r_[np.ones(n, bool), np.zeros(n, bool)]
    t0 = S.t_groups(x, g0[None])[0, 0]
    p0 = S.perm_p(t0, t_ref)
    sw = rng.random((R, n)) < 0.5
    tr_ = S.t_groups(x, np.concatenate([~sw, sw], 1))[:, 0]
    sel["unpaired"].append(bool(p0 < 0.05 and t0 > 0))
    null_sel["unpaired"].append((S.perm_p(tr_, t_ref) < 0.05) & (tr_ > 0))
    p_mf["unpaired"].append(p0)
for k in sel:
    UN["maint_" + k] = sel[k]
    null_sel[k] = np.array(null_sel[k])


def area_table(selcol, null, regcol):
    rows = []
    for r in REG:
        idx = (UN[regcol] == r).to_numpy()
        obs = int(UN.loc[idx, selcol].sum())
        nl = null[idx].sum(0)
        rows.append({"region": r, "n": int(idx.sum()), "selected": obs, "pct": 100 * obs / idx.sum(),
                     "null_mean": nl.mean(), "null_95": np.percentile(nl, 95),
                     "P": (np.sum(nl >= obs) + 1) / (len(nl) + 1),
                     "P_499": (np.sum(nl[:499] >= obs) + 1) / 500})
    return pd.DataFrame(rows)


def chi_hipp(selcol, regcol, target="hippocampus"):
    sub = UN[UN[regcol].isin(REG)]
    a = sub[sub[regcol] == target][selcol]
    b = sub[sub[regcol] != target][selcol]
    tab = np.array([[a.sum(), (~a).sum()], [b.sum(), (~b).sum()]])
    c0 = stats.chi2_contingency(tab, correction=False)
    c1 = stats.chi2_contingency(tab, correction=True)
    return tab, c0[0], c0[1], c1[0], c1[1]


for regcol in ["region_atlas", "region_bundle"]:
    for k in ["paired", "unpaired"]:
        print(f"== regions {regcol}, {k} test")
        print(area_table("maint_" + k, null_sel[k], regcol).round(4).to_string(index=False))
        tab, x0, p0, x1, p1 = chi_hipp("maint_" + k, regcol)
        print(f"   chi-square hippocampus against the other two: {x0:.2f} (P = {p0:.2g}); Yates {x1:.2f} (P = {p1:.2g}); table {tab.tolist()}")
print("per subject, hippocampal maintenance neurons (atlas, paired):")
print(UN[UN.region_atlas == "hippocampus"].groupby("subject").maint_paired.agg(["sum", "count"]).T)

**Result: Reproduced.** With atlas regions and the paired test, 128 of 640 hippocampal neurons (20.0%, paper 20.6 +/- 2.06) are maintenance neurons, against 9.98% in entorhinal cortex and 5.78% in amygdala; chi-square 45.55 (paper 47.30 +/- 4.73), P = 1.5e-11; each area exceeds its scrambled null (P = 0.001 with 1000 repetitions, 0.002 with 499). The unpaired test and the bundle regions give the same pattern (19.2 to 22.2% in hippocampus, chi-square 41.8 to 45.6).

### fig2c-load-dependent-maintenance-neurons

> During maintenance, 15% of hippocampal maintenance neurons (21 neurons) increased their
> firing rate with workload (set size 4 against set sizes 6 and 8), a significant percentage
> against scrambled data; the percentage was not significant in entorhinal cortex or amygdala.

Maintenance neurons of each fig2b reading, correct trials, set size 6 and 8 against 4.
Readings: (a) permutation t test of maintenance rates (2000 permutations); (b) cluster-based
permutation test on rates in 500 ms windows at 20 ms steps (centres -2.75 to -0.25 s, inside
maintenance; 1000 permutations). Both need 6/8 higher at P < 0.05. The null shuffles set-size
labels across trials and reruns the test, 1000 times (P from the first 499 also given).

In [ ]:
rng = np.random.default_rng(202)
R = 1000
res2c = {}
for k in ["paired", "unpaired"]:
    for reading in ["a_ttest", "b_cluster"]:
        obs_sel, null_s = [], []
        for i in np.where(UN["maint_" + k])[0]:
            u = units[i]
            c = u["correct"]
            g = u["set_size"][c] >= 6
            Gr = np.array([rng.permutation(g) for _ in range(R)])
            if reading == "a_ttest":
                x = win_rate(u, S.MAINT)[c][:, None]
                t_ref = S.t_groups(x, S.random_masks(int(g.sum()), len(g), 2000, rng))[:, 0]
                t0 = S.t_groups(x, g[None])[0, 0]
                tr_ = S.t_groups(x, Gr)[:, 0]
                obs_sel.append(bool(S.perm_p(t0, t_ref) < 0.05 and t0 > 0))
                null_s.append((S.perm_p(tr_, t_ref) < 0.05) & (tr_ > 0))
            else:
                X, _ = sliding(u, 0.5, -2.75, -0.25)
                X = X[c]
                p0, pe = S.cluster_test(X, g, 1000, rng, extra=Gr)
                obs_sel.append(bool(p0 < 0.05))
                null_s.append(pe < 0.05)
        idx = np.where(UN["maint_" + k])[0]
        res2c[(k, reading)] = (idx, np.array(obs_sel), np.array(null_s))
        for regcol in ["region_atlas", "region_bundle"]:
            rows = []
            for r in REG:
                m = (UN[regcol].to_numpy()[idx] == r)
                obs = int(np.array(obs_sel)[m].sum())
                nl = np.array(null_s)[m].sum(0)
                rows.append({"region": r, "maintenance": int(m.sum()), "load_dependent": obs,
                             "pct": 100 * obs / max(m.sum(), 1), "null_mean": nl.mean(),
                             "P": (np.sum(nl >= obs) + 1) / (R + 1), "P_499": (np.sum(nl[:499] >= obs) + 1) / 500})
            print(f"== maintenance neurons from the {k} test, reading {reading}, regions {regcol}")
            print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: Reproduced under the criterion.** With the unpaired selection and reading (a), 22 hippocampal maintenance neurons are load dependent (paper 21 +/- 2.1), above the null (P = 0.001), and neither entorhinal cortex (P = 0.32) nor amygdala (P = 0.12) is significant; their percentage, 17.9% of 123, is outside the 15 +/- 1.5% tolerance, which the criterion does not judge. With the paired selection, 20 of 128 (15.6%) are within both tolerances but amygdala is also significant (3 of 23, P = 0.031), which is partial under the criterion.

### fig3e-probe-neurons-entorhinal

> Neurons firing specifically after the probe letter (probe neurons) occurred in all areas,
> most prominently in entorhinal cortex, where 9% of neurons (41) were probe neurons.

Correct trials. Probe rate higher than encoding (-5 to -3 s) and than maintenance (-3 to 0 s),
each by a permutation t test (2000 permutations) at P < 0.05. Probe windows: (a) 0 to 1 s
after probe onset; (b) probe onset to the button press; (c) -500 to 0 ms before the press.
The files store only 2 s after probe onset, so (b) is cut at 2 s and (c) uses the trials whose
press falls within those 2 s. The null shuffles the three period labels within each trial and
reruns the selection (500 times). In the null repetitions the P of each test is taken from
the t distribution instead of 2000 permutations (to keep the run short); the agreement of the
two on the observed data is printed.

In [ ]:
rng = np.random.default_rng(301)
R = 500


def probe_rates(u, reading, c):
    rt = u["rt"]
    if reading == "a":
        return win_rate(u, (0.0, 1.0)), np.ones(len(rt), bool)
    if reading == "b":
        end = np.minimum(rt, 2.0)
        cnt = np.array([np.sum((s >= 0) & (s < e)) for s, e in zip(u["aligned"], end)])
        return cnt / end, np.ones(len(rt), bool)
    ok = rt <= 2.0
    cnt = np.array([np.sum((s >= r - 0.5) & (s < r)) for s, r in zip(u["aligned"], rt)])
    return cnt / 0.5, ok


def ttest_p(a, b, paired):
    if paired:
        t, p = stats.ttest_rel(a, b, axis=-1)
    else:
        t, p = stats.ttest_ind(a, b, axis=-1, equal_var=False)
    return np.nan_to_num(t), np.nan_to_num(p, nan=1.0)


probe_sel = {}
agree = []
for reading in ["a", "b", "c"]:
    for k in ["paired", "unpaired"]:
        obs, nul = [], []
        for u in units:
            pr, ok = probe_rates(u, reading, None)
            c = u["correct"] & ok
            P_, E_, M_ = pr[c], win_rate(u, S.ENC)[c], win_rate(u, S.MAINT)[c]
            n = len(P_)
            sel_ = True
            for other in (E_, M_):
                if k == "paired":
                    d = P_ - other
                    t_ref = S.t_paired_rows(rng.choice([-1.0, 1.0], (2000, n)) * d)
                    t0 = S.t_paired_rows(d[None])[0]
                else:
                    x = np.concatenate([P_, other])[:, None]
                    t_ref = S.t_groups(x, S.random_masks(n, 2 * n, 2000, rng))[:, 0]
                    t0 = S.t_groups(x, np.r_[np.ones(n, bool), np.zeros(n, bool)][None])[0, 0]
                pp = S.perm_p(t0, t_ref)
                tp, ppar = ttest_p(P_, other, k == "paired")
                agree.append((pp < 0.05) == (ppar < 0.05))
                sel_ = sel_ and (pp < 0.05) and (t0 > 0)
            obs.append(sel_)
            V = np.stack([P_, E_, M_], 1)
            perm = np.argsort(rng.random((R, n, 3)), axis=2)
            Vp = np.take_along_axis(np.broadcast_to(V, (R, n, 3)), perm, axis=2)
            t1, p1 = ttest_p(Vp[..., 0], Vp[..., 1], k == "paired")
            t2, p2 = ttest_p(Vp[..., 0], Vp[..., 2], k == "paired")
            nul.append((t1 > 0) & (p1 < 0.05) & (t2 > 0) & (p2 < 0.05))
        UN[f"probe_{reading}_{k}"] = obs
        probe_sel[(reading, k)] = np.array(nul)
print("agreement of permutation and t-distribution P at 0.05 on the observed tests: %.4f" % np.mean(agree))
for regcol in ["region_atlas", "region_bundle"]:
    for reading in ["a", "b", "c"]:
        for k in ["paired", "unpaired"]:
            col = f"probe_{reading}_{k}"
            tab = area_table(col, probe_sel[(reading, k)], regcol)
            _, x0, p0, x1, p1 = chi_hipp(col, regcol, target="entorhinal")
            print(f"== window ({reading}), {k}, regions {regcol}: total probe neurons {int(UN.loc[UN[regcol].isin(REG), col].sum())}")
            print(tab.round(4).to_string(index=False))
            print(f"   chi-square entorhinal against the other two: {x0:.2f} (P = {p0:.2g}); Yates {x1:.2f} (P = {p1:.2g})")

**Result: Reproduced under the criterion with window (b), the probe to the press, and the unpaired test: entorhinal cortex has the highest percentage (44 of 421, 10.**5%; paper 41 +/- 4.1), the chi-square against the other areas is 13.79 (P = 0.0002) and every area exceeds its null (P = 0.002). The chi-square value is outside its tolerance (23.16 +/- 2.32), which the criterion does not judge. Window (a) gives 46 to 48 entorhinal probe neurons with chi-square 20.55 to 21.54; window (c) puts entorhinal cortex below amygdala.

### fig3-probe-onset-not-movement

> Peak firing of probe neurons was higher when aligned to probe onset than when aligned to the
> button press (5.47 +/- 4.97 Hz against 4.28 +/- 4.37 Hz), so their firing reflects the probe
> stimulus more than movement preparation.

Probe neurons of fig3e under each of its readings (the claim depends on it). Per neuron, the
maximum of the trial-averaged rate in 200 ms windows at 20 ms steps, correct trials, after
each alignment, in (a) 0 to 1 s after the event or (b) -1 to +1 s. Button-aligned windows
that fall outside the stored 8 s of a trial are left out of that trial's average (the files
end 2 s after probe onset). Paired permutation t test over neurons (2000 permutations).

In [ ]:
rng = np.random.default_rng(302)


def peak_rates(u, lo, hi):
    c = u["correct"]
    cen = np.round(np.arange(lo, hi + 1e-9, 0.02), 4)
    X, _ = sliding(u, 0.2, lo, hi)
    probe_peak = X[c].mean(0).max()
    rows = []
    for s, r in zip([u["aligned"][i] for i in np.where(c)[0]], u["rt"][c]):
        rel = s - r
        v = []
        for cc in cen:
            a, b = cc - 0.1 + r, cc + 0.1 + r   # window in probe time
            v.append(np.sum((rel >= cc - 0.1) & (rel < cc + 0.1)) / 0.2 if (a >= -6 and b <= 2.0) else np.nan)
        rows.append(v)
    rows = np.array(rows)
    btn = np.nanmean(rows, 0)
    return probe_peak, np.nanmax(btn) if np.isfinite(btn).any() else np.nan, np.isfinite(rows).mean()


for (reading, k) in [("b", "unpaired"), ("a", "paired"), ("a", "unpaired"), ("b", "paired"), ("c", "paired"), ("c", "unpaired")]:
    col = f"probe_{reading}_{k}"
    idx = np.where(UN[col] & UN.region_atlas.isin(REG))[0]
    for wname, (lo, hi) in [("(a) 0 to 1 s", (0.1, 0.9)), ("(b) -1 to 1 s", (-0.9, 0.9))]:
        pk = np.array([peak_rates(units[i], lo, hi) for i in idx])
        ok = np.isfinite(pk[:, 1])
        t, p = S.perm_t(pk[ok, 0], pk[ok, 1], n_perm=2000, paired=True, rng=rng)
        print(f"probe neurons {reading}/{k} (n = {ok.sum()} of {len(idx)}), window {wname}: probe-aligned "
              f"{pk[ok, 0].mean():.2f} +/- {pk[ok, 0].std(ddof=1):.2f} Hz, button-aligned {pk[ok, 1].mean():.2f} +/- "
              f"{pk[ok, 1].std(ddof=1):.2f} Hz; paired t({ok.sum() - 1}) = {t:.2f}, P = {p:.4f}; "
              f"button windows covered {np.mean(pk[:, 2]):.2f}")

**Result: Reproduced under one reading.** With the probe neurons of fig3e window (a), paired test (97 neurons), peaks in 0 to 1 s after the event are 7.72 +/- 5.91 Hz probe-aligned against 6.04 +/- 5.61 Hz button-aligned (paper 5.47 and 4.28, both within tolerance), paired t(96) = 5.07, P = 0.0005. With the probe neurons of the fig3e reading that met its criterion (window (b), unpaired), the difference is not significant (7.26 against 6.76 Hz, P = 0.33), and with -1 to +1 s windows the button-aligned peak is the higher in four of the six readings.

### fig3-probe-neurons-few-task-specific

> Only a few probe neurons reflected the decision (8 neurons, 10.5% of probe neurons, IN
> against OUT), and only one showed workload dependence.

Probe neurons of fig3e. IN against OUT, and set size 4 against 6 and 8, by a cluster-based
permutation test (1000 permutations) at P < 0.05 on rates in 200 ms windows at 20 ms steps
after probe onset, correct trials, over (a) 0 to 1 s or (b) 0 to 2 s (window centres 0.1 to
0.9 s and 0.1 to 1.9 s). A neuron counts for a cluster in either direction.

In [ ]:
rng = np.random.default_rng(303)
for (reading, k) in [("b", "unpaired"), ("a", "paired"), ("a", "unpaired"), ("b", "paired"), ("c", "paired"), ("c", "unpaired")]:
    col = f"probe_{reading}_{k}"
    idx = np.where(UN[col] & UN.region_atlas.isin(REG))[0]
    for wname, (lo, hi) in [("(a) 0-1 s", (0.1, 0.9)), ("(b) 0-2 s", (0.1, 1.9))]:
        n_inout = n_load = 0
        for i in idx:
            u = units[i]
            c = u["correct"]
            X, _ = sliding(u, 0.2, lo, hi)
            X = X[c]
            p_io = min(S.cluster_test(X, u["is_in"][c], 1000, rng, sign=1), S.cluster_test(X, u["is_in"][c], 1000, rng, sign=-1))
            g = u["set_size"][c] >= 6
            p_ld = min(S.cluster_test(X, g, 1000, rng, sign=1), S.cluster_test(X, g, 1000, rng, sign=-1))
            n_inout += p_io < 0.05
            n_load += p_ld < 0.05
        print(f"probe neurons {reading}/{k} ({len(idx)}), {wname}: IN/OUT {n_inout} ({100 * n_inout / len(idx):.1f}%), "
              f"workload {n_load} ({100 * n_load / len(idx):.1f}%)")

**Result: Partially reproduced.** With the probe neurons of fig3e window (b), paired test (107), 8 (7.5%) differ between IN and OUT over 0 to 2 s (paper 8 +/- 0.8, within; 10.5% +/- 1.05, outside), but 7 differ with workload where the paper has one (0 to 2 accepted). Across all readings 2 to 13 probe neurons differ between IN and OUT and 5 to 8 with workload, so the workload count never holds.

## Scalp EEG, hippocampal iEEG and phase locking

Scalp EEG (200 Hz as deposited) and one bipolar iEEG signal per hippocampal macroelectrode
(AH and PH; contact 1 minus contact 2, decimated from 2 kHz to 200 Hz with a zero-phase
filter), from -6 to +2 s around each probe. Trials are pooled over the sessions of a subject
(Fig. 5 legend counts trials of subject 1 over its sessions). Trials marked as artifacts are
left out (Methods, EEG preprocessing). Epochs: last 2 s of maintenance (-2 to 0 s) and
encoding (-5 to -3 s), 400 samples, multitaper with 7 tapers (NW = 4, +/- 2 Hz), 0.5 Hz bins.
The EEG is re-referenced here to the averaged mastoids (A1, A2), as the Methods say; the
deposited EEG is not (checked below). A1 and A2 are not used as scalp sites.

In [ ]:
t0 = time.time()
SIG = S.load_signals(A, workers=20)
t_sig = time.time() - t0
mb_sig = sum((s["ieeg"].shape[0] * 8 * S.IEEG_FS * 2 * s["ieeg"].shape[1] + s["eeg"].size) * 8 for s in SIG.values()) / 1e6
print(f"read {mb_sig:.0f} MB of EEG and iEEG in {t_sig:.0f} s ({mb_sig / t_sig:.1f} MB/s with 20 processes)")

In [ ]:
print("EEG and iEEG samples per trial, sampling intervals and first-sample offsets from trial starts:")
chk = pd.DataFrame([{"path": p, "eeg": s["eeg"].shape, "ieeg_bipolar": s["ieeg"].shape,
                     "eeg_dt": round(s["eeg_dt"], 6), "ieeg_dt": round(s["ieeg_dt"], 6),
                     "max_offset_eeg": s["eeg_start_err"], "max_offset_ieeg": s["ieeg_start_err"],
                     "hipp": ",".join(s["hipp_channels"])} for p, s in SIG.items()])
print(chk.to_string(index=False))
e1 = SIG[A.path[0]]
lab = e1["eeg_labels"]
a1, a2 = e1["eeg"][:, lab.index("A1")], e1["eeg"][:, lab.index("A2")]
print("subject 1 session 1: SD of A1 %.3g, A2 %.3g, A1 + A2 %.3g (file units); if the EEG were re-referenced to the"
      " averaged mastoids, A1 + A2 would be 0" % (a1.std(), a2.std(), (a1 + a2).std()))
print("A1 + A2 relative to A1 - A2, per subject:",
      {d["subject"]: round(float(np.std(SIG[d["path"]]["eeg"][:, SIG[d["path"]]["eeg_labels"].index("A1")] + SIG[d["path"]]["eeg"][:, SIG[d["path"]]["eeg_labels"].index("A2")]) /
                                 np.std(SIG[d["path"]]["eeg"][:, SIG[d["path"]]["eeg_labels"].index("A1")] - SIG[d["path"]]["eeg"][:, SIG[d["path"]]["eeg_labels"].index("A2")])), 2)
       for d in D if d["session"] == "01"})
print("EEG values, 1st and 99th percentiles over all sessions (file says volts): %.1f, %.1f" % tuple(np.percentile(np.concatenate([s["eeg"][:, :2].ravel() for s in SIG.values()]), [1, 99])))
print("bipolar iEEG values, 1st and 99th percentiles (file says volts): %.1f, %.1f" % tuple(np.percentile(np.concatenate([s["ieeg"].ravel() for s in SIG.values()]), [1, 99])))

In [ ]:
FS = 200.0
I_LAST2 = slice(int((6 + S.LAST2[0]) * FS), int((6 + S.LAST2[1]) * FS))
I_ENC = slice(int((6 + S.ENC[0]) * FS), int((6 + S.ENC[1]) * FS))
I_FIX = slice(int((6 + S.FIX[0]) * FS), int((6 + S.FIX[1]) * FS))
SUBJ = {}
for d in D:
    s = SIG[d["path"]]
    tr = d["trials"]
    sj = SUBJ.setdefault(d["subject"], {"eeg": [], "ieeg": [], "trials": [], "eeg_labels": s["eeg_labels"],
                                        "hipp": s["hipp_channels"], "soz": s["hipp_soz"],
                                        "xyz1": s["hipp_xyz1"], "xyz12": s["hipp_xyz12"]})
    assert sj["eeg_labels"] == s["eeg_labels"] and sj["hipp"] == s["hipp_channels"]
    sj["eeg"].append(s["eeg"])
    sj["ieeg"].append(s["ieeg"])
    sj["trials"].append(tr.assign(session=d["session"]))
for sj in SUBJ.values():
    sj["eeg"] = np.concatenate(sj["eeg"])
    # re-reference to the averaged mastoids (Methods, Recording setup); the deposited EEG is not
    ia, ib = sj["eeg_labels"].index("A1"), sj["eeg_labels"].index("A2")
    sj["eeg"] = sj["eeg"] - (sj["eeg"][:, ia:ia + 1] + sj["eeg"][:, ib:ib + 1]) / 2
    sj["ieeg"] = np.concatenate(sj["ieeg"])
    sj["trials"] = pd.concat(sj["trials"], ignore_index=True)
    sj["scalp"] = [i for i, l in enumerate(sj["eeg_labels"]) if l not in ("A1", "A2")]
_, FREQS = S.mt_fourier(np.zeros((1, 400)), FS)
sj1 = SUBJ["01"]


def trial_mask(sj, which, set_sizes=(8,)):
    t = sj["trials"]
    m = ~t.artifact.to_numpy() & t.set_size.isin(set_sizes).to_numpy()
    if which == "correct":
        m &= t.correct.to_numpy()
    return m


def coeffs(sj, mask, period):
    return (S.mt_fourier(sj["ieeg"][mask][:, :, period], FS)[0],
            S.mt_fourier(sj["eeg"][mask][:, :, period], FS)[0])   # tapers x trials x chans x freqs


def plv_pair(Fi, Fe, perm=None):
    if perm is not None:
        Fe = Fe[:, perm]
    return S.plv(Fi, Fe)


def runs(sig, peak):
    """Contiguous run of True bins in `sig` containing index `peak`."""
    a = b = peak
    while a > 0 and sig[a - 1]:
        a -= 1
    while b < len(sig) - 1 and sig[b + 1]:
        b += 1
    return a, b


FMAX = 40.0
FK = FREQS <= FMAX
f40 = FREQS[FK]


def select_pair(sj, which, frange, rng, n_perm=200, fn=S.plv):
    """PLV of every hippocampal-scalp pair, set size 8, last 2 s of maintenance; significance per
    bin against the 95th percentile of PLVs with the trials of the scalp channel permuted
    (Methods, Phase-locking value). Returns the pair and band of the highest significant PLV."""
    m = trial_mask(sj, which)
    Fi, Fe = coeffs(sj, m, I_LAST2)
    Fi, Fe = Fi[..., FK], Fe[..., FK]
    f = FREQS[FK]
    fsel = (f >= frange[0]) & (f <= frange[1])
    n = Fi.shape[1]
    perms = np.array([rng.permutation(n) for _ in range(n_perm)])
    best = None
    table = []
    for hi, hname in enumerate(sj["hipp"]):
        for ei in sj["scalp"]:
            p = fn(Fi[:, :, hi], Fe[:, :, ei])
            null = fn(Fi[:, :, hi][:, :, None, :], Fe[:, perms, ei].transpose(0, 2, 1, 3)) if fn is S.plv else \
                np.array([fn(Fi[:, :, hi], Fe[:, pp, ei]) for pp in perms])
            thr = np.percentile(null, 95, axis=0)
            sig = (p > thr) & fsel
            table.append((hname, sj["eeg_labels"][ei], p[fsel].max(), bool(sig.any()), int(sig.sum())))
            if sig.any():
                k = np.argmax(np.where(sig, p, -1))
                if best is None or p[k] > best["plv"]:
                    a, b = runs(sig, k)
                    best = {"hipp": hname, "hi": hi, "scalp": sj["eeg_labels"][ei], "ei": ei, "plv": float(p[k]),
                            "peak_f": float(f[k]), "band": (float(f[a]), float(f[b])), "n_trials": int(n)}
    return best, pd.DataFrame(table, columns=["hipp", "scalp", "max_plv", "significant", "sig_bins"])


def plv_diff_test(FiA, FeA, FiB, FeB, rng, n_perm=200):
    """PLV(A) - PLV(B) of one pair (coefficients tapers x epochs x freqs) and the 95th percentile
    of differences with the epochs randomly assigned to the two conditions (Methods)."""
    d0 = S.plv(FiA, FeA) - S.plv(FiB, FeB)
    Fi = np.concatenate([FiA, FiB], 1)
    Fe = np.concatenate([FeA, FeB], 1)
    nA = FiA.shape[1]
    null = []
    for _ in range(n_perm):
        p = rng.permutation(Fi.shape[1])
        a, b = p[:nA], p[nA:]
        null.append(S.plv(Fi[:, a], Fe[:, a]) - S.plv(Fi[:, b], Fe[:, b]))
    thr = np.percentile(np.array(null), 95, axis=0)
    return d0, thr, d0 > thr


READINGS = [("correct", (4, 30)), ("correct", (1, 40)), ("all", (4, 30)), ("all", (1, 40))]

### fig6-plv-all-subjects

> All subjects showed a significant hippocampal-scalp PLV peak during maintenance (set size
> 8), with individual frequency bands whose lower bound lay in the theta-alpha range and which
> extended up to 17.5 Hz (median band limits 6.0 to 11.5 Hz).

Readings: trials (a) correct only or (b) all artifact-free; frequency range searched (a) 4 to
30 Hz or (b) 1 to 40 Hz. 200 trial permutations per pair.

In [ ]:
rng = np.random.default_rng(601)
PAIRS = {}
PAIR_TABLES = {}
for which, fr in READINGS:
    rows = []
    for sid, sj in SUBJ.items():
        best, tab = select_pair(sj, which, fr, rng)
        PAIRS[(which, fr, sid)] = best
        PAIR_TABLES[(which, fr, sid)] = tab
        rows.append({"subject": sid, **({k: v for k, v in best.items() if k not in ("hi", "ei")} if best else {"hipp": None}),
                     "pairs_significant": f"{int(tab.significant.sum())}/{len(tab)}"})
    R6 = pd.DataFrame(rows)
    lo = R6.band.dropna().map(lambda b: b[0])
    hi = R6.band.dropna().map(lambda b: b[1])
    print(f"== trials {which}, range {fr} Hz: subjects with a significant pair {R6.hipp.notna().sum()} of {len(R6)}; "
          f"median band limits {np.median(lo):.2f} to {np.median(hi):.2f} Hz; band edges range {lo.min():.1f} to {hi.max():.1f} Hz")
    print(R6.to_string(index=False))
# Variant: PLV with each taper normalized on its own (sternberg.plv_tapers), correct trials, 4 to 30 Hz
rows = []
for sid, sj in SUBJ.items():
    best, tab = select_pair(sj, "correct", (4, 30), rng, fn=S.plv_tapers)
    rows.append({"subject": sid, **{k: v for k, v in best.items() if k not in ("hi", "ei")}})
print("== variant, per-taper PLV, correct trials, 4 to 30 Hz")
print(pd.DataFrame(rows).to_string(index=False))

**Result: Partially reproduced.** All nine subjects have a significant hippocampal-scalp pair under every reading, but the median band limits are 4.0 to 9.5 Hz (correct trials, 4 to 30 Hz), 7.0 to 17.0, 5.5 to 17.5 and 7.5 to 18.0 Hz under the other readings, none within the paper's 6.0 +/- 0.6 to 11.5 +/- 1.15 Hz. Almost every pair is significant somewhere, as the per-bin threshold is uncorrected over bins.

### tables1-plv-per-subject

> Table S1 gives for each subject the scalp electrode of maximal PLV, its peak frequency and
> the maximal PLV (subject 1 Pz 10.5 Hz 0.72; 2 F3 7 Hz 0.40; 3 F3 12 Hz 0.43; 4 F3 14 Hz
> 0.85; 5 P3 5.5 Hz 0.60; 6 O1 7.5 Hz 0.50; 7 O1 7.5 Hz 0.64; 8 O1 11.5 Hz 0.57; 9 O1 8 Hz
> 0.69), with the hippocampal electrode in the left hemisphere for subjects 1 to 6 and the
> right for 7 to 9.

In [ ]:
TS1 = pd.DataFrame({"subject": [f"{i:02d}" for i in range(1, 10)],
                    "scalp": ["Pz", "F3", "F3", "F3", "P3", "O1", "O1", "O1", "O1"],
                    "peak_f": [10.5, 7, 12, 14, 5.5, 7.5, 7.5, 11.5, 8],
                    "plv": [0.72, 0.40, 0.43, 0.85, 0.60, 0.50, 0.64, 0.57, 0.69],
                    "hemi": ["L"] * 6 + ["R"] * 3})
for which, fr in READINGS:
    rows = []
    for r in TS1.itertuples():
        b = PAIRS[(which, fr, r.subject)]
        if b is None:
            rows.append({"subject": r.subject, "match": False})
            continue
        same_e = b["scalp"] == r.scalp
        same_h = b["hipp"][-1] == r.hemi
        in_tol = abs(b["plv"] - r.plv) <= 0.1 * r.plv
        rows.append({"subject": r.subject, "scalp": b["scalp"], "paper_scalp": r.scalp, "hipp": b["hipp"],
                     "paper_hemi": r.hemi, "plv": round(b["plv"], 3), "paper_plv": r.plv,
                     "peak_f": b["peak_f"], "paper_f": r.peak_f, "electrode": same_e, "hemisphere": same_h,
                     "plv_in_tol": in_tol, "match": bool(same_e and same_h and in_tol)})
    R = pd.DataFrame(rows)
    print(f"== trials {which}, range {fr}: subjects matching on electrode, hemisphere and PLV: {int(R.match.sum())} of 9;"
          f" electrode {int(R.electrode.sum())}, hemisphere {int(R.hemisphere.sum())}, PLV within 10% {int(R.plv_in_tol.sum())}")
    print(R.to_string(index=False))
# Is the paper's electrode present in each subject's EEG?
print("paper's scalp electrode present in the deposited EEG:",
      {r.subject: r.scalp in SUBJ[r.subject]["eeg_labels"] for r in TS1.itertuples()})

**Result: Not reproduced.** No subject matches table S1 on scalp electrode, hemisphere and maximal PLV together under any reading; the electrode matches in at most 1 subject, the hemisphere in 4 to 6, and the PLV within 10% in 2 to 3. With the PLV the Methods define, values are in the paper's range (0.47 to 0.88).

### fig6-plv-workload-eight-of-nine

> During maintenance, PLV in the individual frequency band was higher for set size 8 than set
> size 4 in eight of nine subjects (range 5.5 to 16.5 Hz, median band limits 7.8 to 10 Hz).

The pair of fig6-plv-all-subjects under each of its readings. PLV(8) - PLV(4) against the 95th
percentile of 200 differences with the epochs randomly assigned to the two set sizes. A
subject counts when a significant bin falls in the individual band. The workload band is the
contiguous run of significant bins with the largest difference inside the individual band.

In [ ]:
rng = np.random.default_rng(602)
f40 = FREQS[FK]
WL = {}
for which, fr in READINGS:
    rows = []
    for sid, sj in SUBJ.items():
        b = PAIRS[(which, fr, sid)]
        m8, m4 = trial_mask(sj, which, (8,)), trial_mask(sj, which, (4,))
        Fi8, Fe8 = coeffs(sj, m8, I_LAST2)
        Fi4, Fe4 = coeffs(sj, m4, I_LAST2)
        d, thr, sig = plv_diff_test(Fi8[:, :, b["hi"]][..., FK], Fe8[:, :, b["ei"]][..., FK],
                                    Fi4[:, :, b["hi"]][..., FK], Fe4[:, :, b["ei"]][..., FK], rng)
        inband = (f40 >= b["band"][0]) & (f40 <= b["band"][1])
        counts = bool((sig & inband).any())
        wb = None
        if counts:
            k = np.argmax(np.where(sig & inband, d, -np.inf))
            a, e = runs(sig, k)
            wb = (float(f40[a]), float(f40[e]))
        WL[(which, fr, sid)] = (d, thr, sig)
        rows.append({"subject": sid, "pair": f'{b["hipp"]}-{b["scalp"]}', "band": b["band"], "n8": int(m8.sum()),
                     "n4": int(m4.sum()), "max_diff_in_band": float(d[inband].max()), "counts": counts, "workload_band": wb})
    R = pd.DataFrame(rows)
    wbs = R.workload_band.dropna()
    print(f"== trials {which}, range {fr}: subjects with higher PLV for set size 8: {int(R.counts.sum())} of 9"
          + (f"; median workload band limits {np.median([w[0] for w in wbs]):.2f} to {np.median([w[1] for w in wbs]):.2f} Hz,"
             f" range {min(w[0] for w in wbs):.1f} to {max(w[1] for w in wbs):.1f} Hz" if len(wbs) else ""))
    print(R.round(3).to_string(index=False))

**Result: Partially reproduced.** With all trials and 1 to 40 Hz, eight of nine subjects have a higher set size 8 PLV in their individual band, as the paper says, but the workload bands (median 10.75 to 11.0 Hz, range 2.0 to 34.0 Hz) differ from the paper's (7.8 to 10 Hz, 5.5 to 16.5 Hz). Under the other readings 5 to 7 subjects count.

### fig6-plv-maintenance-over-encoding

> The PLV during the last 2 s of maintenance exceeded the PLV during encoding in eight of nine
> subjects, so hippocampal-cortical connectivity in the theta-alpha range increased
> specifically during maintenance.

The pair of fig6-plv-all-subjects. PLV(last 2 s of maintenance) - PLV(encoding, -5 to -3 s),
against the 95th percentile of 200 differences with epochs randomly assigned to the two
periods. Trials (a) set size 8 or (b) all set sizes.

In [ ]:
rng = np.random.default_rng(603)
ME = {}
for which, fr in READINGS:
    for sets in [(8,), (4, 6, 8)]:
        rows = []
        for sid, sj in SUBJ.items():
            b = PAIRS[(which, fr, sid)]
            m = trial_mask(sj, which, sets)
            FiM, FeM = coeffs(sj, m, I_LAST2)
            FiE, FeE = coeffs(sj, m, I_ENC)
            d, thr, sig = plv_diff_test(FiM[:, :, b["hi"]][..., FK], FeM[:, :, b["ei"]][..., FK],
                                        FiE[:, :, b["hi"]][..., FK], FeE[:, :, b["ei"]][..., FK], rng)
            inband = (f40 >= b["band"][0]) & (f40 <= b["band"][1])
            ME[(which, fr, sets, sid)] = (d, thr, sig)
            rows.append({"subject": sid, "pair": f'{b["hipp"]}-{b["scalp"]}', "band": b["band"], "n": int(m.sum()),
                         "max_diff_in_band": float(d[inband].max()), "counts": bool((sig & inband).any())})
        R = pd.DataFrame(rows)
        print(f"== trials {which}, range {fr}, set sizes {sets}: subjects with maintenance PLV above encoding: {int(R.counts.sum())} of 9")
        print(R.round(3).to_string(index=False))

**Result: Not reproduced.** Maintenance PLV exceeds encoding PLV in the individual band in 4 to 6 of nine subjects under the readings (paper eight).

### fig6-power-unchanged-individual-band

> In the majority of subjects, iEEG and EEG power in the individual frequency band did not
> differ during maintenance, which suggests that the PLV differences came from phase
> consistency.

A claim of absence. Per subject, the two channels of the maximal PLV pair, last 2 s of
maintenance, multitaper power, set size 8 against 4, cluster-based permutation test (1000
permutations, either sign) over the bins of the individual band, P < 0.05.

In [ ]:
rng = np.random.default_rng(604)


def power_epochs(sj, mask, chan_kind, ci, period=I_LAST2):
    x = (sj["ieeg"] if chan_kind == "ieeg" else sj["eeg"])[mask][:, ci, period]
    P, _ = S.mt_power(x, FS)
    return P[:, FK]


def power_diff(sj, which, chan_kind, ci, band, rng):
    m8, m4 = trial_mask(sj, which, (8,)), trial_mask(sj, which, (4,))
    P = np.concatenate([power_epochs(sj, m8, chan_kind, ci), power_epochs(sj, m4, chan_kind, ci)])
    g = np.r_[np.ones(m8.sum(), bool), np.zeros(m4.sum(), bool)]
    inb = (f40 >= band[0]) & (f40 <= band[1])
    X = P[:, inb]
    p_up = S.cluster_test(X, g, 1000, rng, sign=1)
    p_dn = S.cluster_test(X, g, 1000, rng, sign=-1)
    return p_up, p_dn, float(np.log10(X[g].mean() / X[~g].mean()))


for which, fr in READINGS:
    rows = []
    for sid, sj in SUBJ.items():
        b = PAIRS[(which, fr, sid)]
        e_up, e_dn, e_eff = power_diff(sj, which, "eeg", b["ei"], b["band"], rng)
        i_up, i_dn, i_eff = power_diff(sj, which, "ieeg", b["hi"], b["band"], rng)
        rows.append({"subject": sid, "pair": f'{b["hipp"]}-{b["scalp"]}', "band": b["band"],
                     "EEG_P_up": e_up, "EEG_P_down": e_dn, "EEG_log10_ratio": e_eff,
                     "iEEG_P_up": i_up, "iEEG_P_down": i_dn, "iEEG_log10_ratio": i_eff})
    R = pd.DataFrame(rows)
    ne = int(((R.EEG_P_up >= 0.05) & (R.EEG_P_down >= 0.05)).sum())
    ni = int(((R.iEEG_P_up >= 0.05) & (R.iEEG_P_down >= 0.05)).sum())
    print(f"== trials {which}, range {fr}: subjects without an EEG power difference {ne} of 9, without an iEEG difference {ni} of 9")
    print(R.round(4).to_string(index=False))

**Result: Not reproduced, resting on an assumption.** Under the adopted reading (correct trials, 4 to 30 Hz), 4 of nine subjects show no EEG power difference (five needed) and 7 no iEEG difference. With 1 to 40 Hz the counts are 6 and 8, which would reproduce the claim; with all trials 4 and 5 to 7.

### fig5-subject1-eeg-power-workload

> In subject 1, scalp EEG power during the last 2 s of maintenance was higher for set size 8
> than for set size 4 at Pz (8.5 to 16 Hz) and at Fz (8 to 12.5 Hz); this workload-dependent
> power increase did not appear in the other subjects.

Last 2 s of maintenance, multitaper power, set size 8 against 4, cluster-based permutation
test over frequency (1000 permutations), ranges (a) 4 to 30 Hz, (b) 1 to 40 Hz.

In [ ]:
rng = np.random.default_rng(605)


def power_clusters(sj, which, chan_kind, ci, fr, rng, sign=1, baseline=False):
    m8, m4 = trial_mask(sj, which, (8,)), trial_mask(sj, which, (4,))
    P8, P4 = power_epochs(sj, m8, chan_kind, ci), power_epochs(sj, m4, chan_kind, ci)
    if baseline:
        def fixp(mask):
            x = (sj["ieeg"] if chan_kind == "ieeg" else sj["eeg"])[mask][:, ci, I_FIX]
            F, fq = S.mt_fourier(x, FS, nw=2.0, k=3)     # 1 s epoch, +/- 2 Hz, 3 tapers
            pw = (np.abs(F) ** 2).mean(0)
            return np.array([np.interp(f40, fq, r) for r in pw])
        P8, P4 = P8 / fixp(m8), P4 / fixp(m4)
    sel = (f40 >= fr[0]) & (f40 <= fr[1])
    X = np.concatenate([P8, P4])[:, sel]
    g = np.r_[np.ones(len(P8), bool), np.zeros(len(P4), bool)]
    from scipy import stats as st_
    thr = st_.t.ppf(0.975, len(g) - 2)
    G = S.random_masks(int(g.sum()), len(g), 1000, rng)
    Tn = S.t_groups(X, G)
    null = np.sort(np.maximum(S.cluster_max(Tn, thr, 1), S.cluster_max(Tn, thr, -1)))
    T0 = S.t_groups(X, g[None])
    _, cl = S.cluster_masses(T0, thr)
    fs_ = f40[sel]
    out = []
    for a, b, mss in cl:
        if np.sign(mss) != sign:
            continue
        p = (np.sum(null >= abs(mss)) + 1) / (len(null) + 1)
        out.append((float(fs_[a]), float(fs_[b - 1]), round(mss, 1), p))
    return out, int(m8.sum()), int(m4.sum())


for which, fr in READINGS:
    for el, target in [("Pz", (8.5, 16)), ("Fz", (8, 12.5))]:
        cl, n8, n4 = power_clusters(sj1, which, "eeg", sj1["eeg_labels"].index(el), fr, rng)
        hit = any(p < 0.05 and a <= target[1] and b >= target[0] for a, b, _, p in cl)
        print(f"subject 1, trials {which}, range {fr}, {el} (n8 = {n8}, n4 = {n4}): positive clusters (from, to Hz, mass, P) {cl}; "
              f"significant and overlapping {target}: {hit}")
print("other subjects, set size 8 above 4 (significant positive clusters):")
for which, fr in READINGS:
    rows = []
    for sid, sj in SUBJ.items():
        if sid == "01":
            continue
        b = PAIRS[(which, fr, sid)]
        cl, _, _ = power_clusters(sj, which, "eeg", b["ei"], fr, rng)
        cln, _, _ = power_clusters(sj, which, "eeg", b["ei"], fr, rng, sign=-1)
        r = {"subject": sid, "electrode": b["scalp"], "increase": [c for c in cl if c[3] < 0.05],
             "decrease": [c for c in cln if c[3] < 0.05]}
        for el in ("Pz", "Fz"):
            if el in sj["eeg_labels"]:
                c2, _, _ = power_clusters(sj, which, "eeg", sj["eeg_labels"].index(el), fr, rng)
                r[f"{el}_increase"] = [c for c in c2 if c[3] < 0.05]
        rows.append(r)
    R = pd.DataFrame(rows)
    print(f"== trials {which}, range {fr}: subjects 2 to 9 with a significant increase at the maximal-PLV electrode: "
          f"{int((R.increase.map(len) > 0).sum())}")
    print(R.to_string(index=False))

**Result: Partially reproduced.** In subject 1, set size 8 power exceeds set size 4 at Pz (cluster 8.5 to 14.5 Hz, P = 0.002) and Fz (8 to 14 Hz, P = 0.002), as the paper shows, under every reading. But other subjects show increases too: at their maximal-PLV electrode, subjects 2 and 6 with correct trials and 4 to 30 Hz, and 2 to 4 subjects under the other readings.

### fig5-subject1-hippocampal-power-workload

> In subject 1, theta-alpha power (7.5 to 10 Hz) in the left hippocampal iEEG increased with
> workload during maintenance.

Subject 1, the hippocampal channel of the maximal PLV pair, last 2 s of maintenance, power
normalized to the fixation power of the same trial (Fig. 5H legend; the 1 s fixation epoch
with 3 tapers, +/- 2 Hz, interpolated to the 0.5 Hz bins), set size 8 against 4, cluster
test at P < 0.05. Without the normalization as a variant. All subjects' channels reported.

In [ ]:
rng = np.random.default_rng(606)
for which, fr in READINGS:
    b = PAIRS[(which, fr, "01")]
    for base in (True, False):
        cl, n8, n4 = power_clusters(sj1, which, "ieeg", b["hi"], fr, rng, baseline=base)
        hit = any(p < 0.05 and a <= 10 and bb >= 7.5 for a, bb, _, p in cl)
        print(f"subject 1, trials {which}, range {fr}, {b['hipp']} (normalized {base}): positive clusters {cl}; overlapping 7.5-10 Hz and significant: {hit}")
    for h in sj1["hipp"]:
        cl, _, _ = power_clusters(sj1, which, "ieeg", sj1["hipp"].index(h), fr, rng, baseline=True)
        print(f"   channel {h}: significant positive clusters {[c for c in cl if c[3] < 0.05]}")
print("all subjects, maximal-PLV hippocampal channel, normalized, trials correct, 4 to 30 Hz:")
for sid, sj in SUBJ.items():
    b = PAIRS[("correct", (4, 30), sid)]
    cl, _, _ = power_clusters(sj, "correct", "ieeg", b["hi"], (4, 30), rng, baseline=True)
    print(f"   {sid} {b['hipp']}: significant increases {[c for c in cl if c[3] < 0.05]}")

**Result: Reproduced.** In subject 1 the left posterior hippocampal channel (PHL, the hippocampal channel of the maximal pair) has higher fixation-normalized power for set size 8 in 7.5 to 9.5 Hz (P = 0.033, correct trials, 4 to 30 Hz) and 7.5 to 10 Hz with all trials; without normalization the cluster is significant only with all trials.

### fig5-subject1-neurons-workload

> In subject 1, the average firing rate of the five hippocampal maintenance neurons was higher
> for high workload (set size 8 against 4) during encoding and maintenance.

Hippocampal maintenance neurons of subject 1 (fig2b, each reading) with more than 100 spikes in
encoding and more than 100 in maintenance (summed over correct trials). Rate in 500 ms windows
at 20 ms steps (centres -4.75 to -0.25 s), set size 8 against 4, correct trials,
cluster-based permutation test (1000 permutations). Unit of the test: (a) trials, the rate
averaged over the neurons of the session (subject 1's neurons come from two sessions, so trials
of both sessions are pooled), (b) neuron-trial pairs.

In [ ]:
rng = np.random.default_rng(607)
for k in ["paired", "unpaired"]:
    for regcol in ["region_atlas", "region_bundle"]:
        idx = [i for i in np.where((UN.subject == "01") & (UN[regcol] == "hippocampus") & UN["maint_" + k])[0]]
        keep = [i for i in idx if win_counts(units[i], S.ENC)[units[i]["correct"]].sum() > 100
                and win_counts(units[i], S.MAINT)[units[i]["correct"]].sum() > 100]
        line = f"maintenance neurons {k}, regions {regcol}: {len(idx)} in subject 1 hippocampus, {len(keep)} with > 100 spikes in both periods"
        if not keep:
            print(line)
            continue
        Xs, gs, Xn, gn = [], [], [], []
        for sess_ in sorted(set(UN.sess[keep])):
            ii = [i for i in keep if UN.sess[i] == sess_]
            c = units[ii[0]]["correct"] & np.isin(units[ii[0]]["set_size"], (4, 8))
            Xm = np.mean([sliding(units[i], 0.5, -4.75, -0.25)[0][c] for i in ii], axis=0)
            Xs.append(Xm)
            gs.append(units[ii[0]]["set_size"][c] == 8)
            for i in ii:
                Xn.append(sliding(units[i], 0.5, -4.75, -0.25)[0][c])
                gn.append(units[i]["set_size"][c] == 8)
        cen = np.round(np.arange(-4.75, -0.25 + 1e-9, 0.02), 4)
        for name, X, g in [("(a) trials", np.concatenate(Xs), np.concatenate(gs)), ("(b) neuron-trials", np.concatenate(Xn), np.concatenate(gn))]:
            from scipy import stats as st_
            thr = st_.t.ppf(0.975, len(g) - 2)
            G = S.random_masks(int(g.sum()), len(g), 1000, rng)
            Tn = S.t_groups(X, G)
            null = np.sort(np.maximum(S.cluster_max(Tn, thr, 1), S.cluster_max(Tn, thr, -1)))
            _, cl = S.cluster_masses(S.t_groups(X, g[None]), thr)
            sig = [(float(cen[a]), float(cen[b - 1]), round(m, 1), (np.sum(null >= abs(m)) + 1) / 1001) for a, b, m in cl if m > 0]
            sig = [c for c in sig if c[3] < 0.05]
            enc = any(a < -3 for a, b, _, _ in sig)
            mnt = any(b > -3 for a, b, _, _ in sig)
            print(line + f"; {name} (n = {len(g)}, {int(g.sum())} of set size 8): significant positive clusters {sig}; "
                  f"in encoding {enc}, in maintenance {mnt}")

**Result: Partially reproduced.** Subject 1 has 14 hippocampal maintenance neurons with more than 100 spikes in both periods (paper 5, tolerance 0). Their rate is higher for set size 8 in a cluster from -4.67 to -1.57 s, which spans encoding and maintenance (P = 0.001), under every reading.

### fig5-subject1-plv-hippocampus-pz

> In subject 1, the task increased the phase locking value (PLV) between hippocampal iEEG and
> scalp EEG at Pz in the alpha band during the last 2 s of maintenance; this PLV was higher
> for set size 8 than 4 (9.0 to 12.0 Hz) and maximal at occipitoparietal electrodes. The PLV
> to Fz was also higher for set size 8 (8.5 to 10 Hz).

The selection, workload and maintenance-encoding tests of the fig6 claims, for subject 1, with
the left hippocampal channel of the maximal pair and with each hippocampal channel against Pz
and Fz.

In [ ]:
rng = np.random.default_rng(608)


def pair_tests(sj, which, hi, ei, rng):
    m8, m4 = trial_mask(sj, which, (8,)), trial_mask(sj, which, (4,))
    Fi8, Fe8 = coeffs(sj, m8, I_LAST2)
    Fi4, Fe4 = coeffs(sj, m4, I_LAST2)
    FiE, FeE = coeffs(sj, m8, I_ENC)
    a = lambda F, c: F[:, :, c][..., FK]
    dW, tW, sW = plv_diff_test(a(Fi8, hi), a(Fe8, ei), a(Fi4, hi), a(Fe4, ei), rng)
    dM, tM, sM = plv_diff_test(a(Fi8, hi), a(Fe8, ei), a(FiE, hi), a(FeE, ei), rng)
    return sW, sM, S.plv(a(Fi8, hi), a(Fe8, ei))


def bins_in(sig, lo, hi):
    return bool((sig & (f40 >= lo) & (f40 <= hi)).any())


def sig_ranges(sig):
    out, i = [], 0
    while i < len(sig):
        if sig[i]:
            j = i
            while j < len(sig) and sig[j]:
                j += 1
            out.append((float(f40[i]), float(f40[j - 1])))
            i = j
        else:
            i += 1
    return out


for which, fr in READINGS:
    b = PAIRS[(which, fr, "01")]
    print(f"== subject 1, trials {which}, range {fr}: maximal pair {b['hipp']}-{b['scalp']} at {b['peak_f']} Hz (PLV {b['plv']:.3f})")
for which in ["correct", "all"]:
    for h in sj1["hipp"]:
        hi = sj1["hipp"].index(h)
        res = {}
        for el in ["Pz", "Fz"]:
            sW, sM, p8 = pair_tests(sj1, which, hi, sj1["eeg_labels"].index(el), rng)
            res[el] = (sW, sM, p8)
        alpha = (f40 >= 8) & (f40 <= 12)
        top = sorted([(float(S.plv(coeffs(sj1, trial_mask(sj1, which), I_LAST2)[0][:, :, hi][..., FK],
                                   coeffs(sj1, trial_mask(sj1, which), I_LAST2)[1][:, :, ei][..., FK])[alpha].mean()), sj1["eeg_labels"][ei])
                      for ei in sj1["scalp"]], reverse=True)[:4]
        print(f"trials {which}, {h}: Pz maint > enc in 8-10 Hz {bins_in(res['Pz'][1], 8, 10)} (significant {sig_ranges(res['Pz'][1])}); "
              f"Pz workload in 9-12 Hz {bins_in(res['Pz'][0], 9, 12)} ({sig_ranges(res['Pz'][0])}); "
              f"Fz workload in 8.5-10 Hz {bins_in(res['Fz'][0], 8.5, 10)} ({sig_ranges(res['Fz'][0])}); "
              f"highest mean 8-12 Hz PLV electrodes {[(e, round(v, 3)) for v, e in top]}")

**Result: Partially reproduced.** The maximal pair of subject 1 is the left hippocampal channel PHL with O2 (PLV 0.741 at 9 Hz), not Pz, which is second (mean alpha PLV 0.626 against 0.635). For PHL with Pz, maintenance PLV exceeds encoding in 8 to 11.5 Hz, the set size 8 PLV exceeds set size 4 in 8.5 to 10 Hz, the highest alpha PLV is at occipitoparietal sites (O2, Pz, O1, T6), and the PLV to Fz is higher for set size 8 in 9.5 to 10.5 Hz.

### fig6i-plv-posterior-gradient

> Across subjects, maximal PLV increased from the anterior to the posterior end of the
> hippocampal axis (34 electrode pairs, R2 = 0.118), and PLV to posterior sites (y < -20 mm)
> was higher than to anterior sites.

Per subject, the scalp electrode of the maximal pair with every hippocampal bipolar channel;
the maximal PLV in the individual band (set size 8, last 2 s), against the anterior-posterior
coordinate. The files give coordinates with +x posterior; MNI y is taken as -x (assumption:
the axes are described in the electrodes table, and AH contacts lie at x 11 to 21 mm, PH at 22
to 30 mm, see the data check). Coordinate of a channel: (a) contact 1, (b) midpoint of
contacts 1 and 2. Fit P from 2000 permutations of the PLV values over positions (two-sided on
the slope); posterior against anterior by an unpaired permutation t test.

In [ ]:
rng = np.random.default_rng(609)
GRAD = {}
for which, fr in READINGS:
    rows = []
    for sid, sj in SUBJ.items():
        b = PAIRS[(which, fr, sid)]
        Fi, Fe = coeffs(sj, trial_mask(sj, which), I_LAST2)
        inb = (f40 >= b["band"][0]) & (f40 <= b["band"][1])
        for hi, h in enumerate(sj["hipp"]):
            p = S.plv(Fi[:, :, hi][..., FK], Fe[:, :, b["ei"]][..., FK])
            rows.append({"subject": sid, "hipp": h, "scalp": b["scalp"], "plv": float(p[inb].max()),
                         "y_a": -float(sj["xyz1"][hi][0]), "y_b": -float(sj["xyz12"][hi][0]),
                         "side": h[-1], "soz": sj["soz"][hi], "is_max": h == b["hipp"]})
    G = pd.DataFrame(rows)
    GRAD[(which, fr)] = G
    for yc in ["y_a", "y_b"]:
        y, x = G.plv.to_numpy(), G[yc].to_numpy()
        sl, ic, r, p, _ = stats.linregress(-x, y)          # slope per mm toward posterior
        null = np.array([stats.linregress(-x, rng.permutation(y)).slope for _ in range(2000)])
        pp = (np.sum(np.abs(null) >= abs(sl)) + 1) / 2001
        post, ant = y[x < -20], y[x >= -20]
        t, pt = S.perm_t(post, ant, n_perm=2000, rng=rng)
        print(f"trials {which}, range {fr}, coordinate {yc}: {len(G)} pairs; slope {sl:.4f} per mm posterior, R2 = {r ** 2:.3f}, "
              f"permutation P = {pp:.4f}; posterior (n = {len(post)}) {post.mean():.3f} against anterior (n = {len(ant)}) "
              f"{ant.mean():.3f}, t = {t:.2f}, P = {pt:.4f}")
print("y range of the channels (mm):", GRAD[READINGS[0]].y_a.min().round(1), "to", GRAD[READINGS[0]].y_a.max().round(1))

**Result: Not reproduced.** The dataset gives 31 hippocampal channels (paper 34). PLV rises toward posterior sites (slope 0.0064 per mm), but R2 = 0.045 (paper 0.118) and permutation P = 0.26; posterior sites are higher than anterior ones without significance (0.501 against 0.385, P = 0.14). No reading is significant.

### fig6i-no-left-right-difference

> There was no evidence that the PLV differed between left and right hippocampus.

A claim of absence: the PLV values of fig6i-plv-posterior-gradient, left against right
channels, unpaired permutation t test (2000 permutations), P < 0.05.

In [ ]:
rng = np.random.default_rng(610)
for which, fr in READINGS:
    G = GRAD[(which, fr)]
    l, r = G[G.side == "L"].plv, G[G.side == "R"].plv
    t, p = S.perm_t(l, r, n_perm=2000, rng=rng)
    print(f"trials {which}, range {fr}: left {l.mean():.3f} (n = {len(l)}), right {r.mean():.3f} (n = {len(r)}), "
          f"difference {l.mean() - r.mean():+.3f}, t = {t:.2f}, P = {p:.4f}")

**Result: Reproduced.** Left and right hippocampal channels do not differ (0.464 against 0.431, P = 0.65, correct trials, 4 to 30 Hz; P = 0.33 to 0.78 under the other readings).

### soz-plv-no-difference

> The increase in PLV did not differ significantly between hippocampal channels inside and
> outside the seizure onset zone.

A claim of absence. SOZ membership of a bipolar channel: either of its contacts marked
`is_inside_soz`. Increase (a) set size 8 minus 4, (b) maintenance minus encoding, the maximum
in the individual band; channels (a) all hippocampal channels with the maximal-PLV scalp
electrode, (b) the nine maximal pairs. Unpaired permutation t test (2000), P < 0.05.

In [ ]:
rng = np.random.default_rng(611)
for which, fr in READINGS:
    rows = []
    for sid, sj in SUBJ.items():
        b = PAIRS[(which, fr, sid)]
        m8, m4 = trial_mask(sj, which, (8,)), trial_mask(sj, which, (4,))
        Fi8, Fe8 = coeffs(sj, m8, I_LAST2)
        Fi4, Fe4 = coeffs(sj, m4, I_LAST2)
        FiE, FeE = coeffs(sj, m8, I_ENC)
        inb = (f40 >= b["band"][0]) & (f40 <= b["band"][1])
        for hi, h in enumerate(sj["hipp"]):
            a = lambda F, c: F[:, :, c][..., FK]
            p8 = S.plv(a(Fi8, hi), a(Fe8, b["ei"]))
            rows.append({"subject": sid, "hipp": h, "soz": sj["soz"][hi], "is_max": h == b["hipp"],
                         "inc_load": float((p8 - S.plv(a(Fi4, hi), a(Fe4, b["ei"])))[inb].max()),
                         "inc_period": float((p8 - S.plv(a(FiE, hi), a(FeE, b["ei"])))[inb].max())})
    Z = pd.DataFrame(rows)
    for chans, sub_ in [("(a) all channels", Z), ("(b) nine maximal pairs", Z[Z.is_max])]:
        for inc in ["inc_load", "inc_period"]:
            s_, n_ = sub_[sub_.soz][inc], sub_[~sub_.soz][inc]
            if len(s_) < 2 or len(n_) < 2:
                print(f"trials {which}, range {fr}, {chans}, {inc}: too few channels (SOZ {len(s_)}, non-SOZ {len(n_)})")
                continue
            t, p = S.perm_t(s_, n_, n_perm=2000, rng=rng)
            print(f"trials {which}, range {fr}, {chans}, {inc}: SOZ {s_.mean():.3f} (n = {len(s_)}), non-SOZ {n_.mean():.3f} "
                  f"(n = {len(n_)}), difference {s_.mean() - n_.mean():+.3f}, t = {t:.2f}, P = {p:.4f}")

**Result: Reproduced.** No reading finds a difference between SOZ and non-SOZ channels (differences -0.134 to +0.056, P = 0.12 to 0.97).

## Population decoding

Pseudopopulation decoding as in the Methods (Neural decoding of firing rates) with the NDT
defaults the Methods leave open (see `sternberg.decode`). Features: rates in a 1500 ms window
at 250 ms steps; reading (a) a boxcar (the main reading), (b) a Gaussian with SD 250 ms
(the window as +/- 3 SD; standard choice). Normalization: (a) z-score on training data,
(b) none. Neurons enter when they have enough trials of each class for the draws. Accuracy:
reading (a) per run the mean over the steps inside the period, then the median over runs
(main), (b) median over runs and steps together. P against the null runs with scrambled
labels, (k + 1) / (n + 1), then times 12 (Bonferroni, Methods). The Methods use 500
resampling runs and 500 null runs; to keep the notebook within its time limit this attempt uses
100 runs and 250 null runs (the smallest corrected P is then 12/251 = 0.048). Regions by the atlas label
(main) and by the bundle name.

In [ ]:
CENTERS = {"encoding": np.arange(-4.75, -3.25 + 1e-9, 0.25), "maintenance": np.arange(-2.75, -0.25 + 1e-9, 0.25)}


def feats_for(u, centres, kind="boxcar"):
    if kind == "boxcar":
        X, _ = None, None
        cs = np.concatenate([np.zeros((u["bins"].shape[0], 1)), np.cumsum(u["bins"], 1)], 1)
        out = []
        for c in centres:
            a = int(round((c - 0.75 + 6) / 0.02))
            b = int(round((c + 0.75 + 6) / 0.02))
            a, b = max(a, 0), min(b, u["bins"].shape[1])
            out.append((cs[:, b] - cs[:, a]) / ((b - a) * 0.02))
        return np.array(out).T
    t = np.round(np.arange(-6.0, 2.0, 0.02) + 0.01, 4)
    out = []
    for c in centres:
        w = np.exp(-0.5 * ((t - c) / 0.25) ** 2) * (np.abs(t - c) <= 0.75)
        out.append(u["bins"] @ (w / (w.sum() * 0.02)))
    return np.array(out).T


def build(pop_idx, label_fn, need, centres, kind="boxcar"):
    """Features and labels of the neurons in pop_idx that have at least need[c] trials per class."""
    F, L, used = [], [], []
    for i in pop_idx:
        u = units[i]
        y, keep = label_fn(u)
        if np.sum(y[keep] == 0) < need[0] or np.sum(y[keep] == 1) < need[1]:
            continue
        F.append(feats_for(u, centres, kind)[keep])
        L.append(y[keep].astype(int))
        used.append(i)
    return F, L, used


def summarize(acc, null, n_tests=12):
    a_med = np.median(acc.mean(1))
    b_med = np.median(acc)
    pa = (np.sum(null.mean(1) >= a_med) + 1) / (len(null) + 1)
    pb = (np.sum(null.ravel() >= b_med) + 1) / (null.size + 1)
    return {"median_a": 100 * a_med, "median_b": 100 * b_med, "null_median": 100 * np.median(null.mean(1)),
            "null_95": 100 * np.percentile(null.mean(1), 95), "P_a": pa, "P_a_bonf": min(1, pa * n_tests),
            "P_b": pb, "P_b_bonf": min(1, pb * n_tests)}


def region_idx(r, regcol="region_atlas", extra=None):
    m = (UN[regcol] == r) if r != "all" else UN[regcol].isin(REG)
    if extra is not None:
        m &= extra
    return list(np.where(m)[0])

### fig4e-maintenance-decodes-workload

> During maintenance, workload (set size 4 against 6 and 8) could be decoded from hippocampal
> neurons with a median accuracy of 80%, but not from neurons of the other areas.

Correct trials, set size 4 (class 0) against 6 and 8 (class 1), 10 splits, steps -2.75 to
-0.25 s. Class balance: (a) 10 trials per class (NDT), (b) 10 of set size 4 and 20 of set
sizes 6 and 8, the design's 1:2 ratio (the "all trials" reading as far as a pseudopopulation
allows; chance is then 67% for a classifier that always answers "high").

In [ ]:
rng = np.random.default_rng(401)
NR, NN = 100, 250   # Methods: 500 and 500; reduced for the run time (minimum P 1/251, x 12 = 0.048)
lab_load = lambda u: ((u["set_size"] >= 6).astype(int), u["correct"])
DEC4E = {}
for regcol in ["region_atlas", "region_bundle"]:
    for r in REG:
        F, L, used = build(region_idx(r, regcol), lab_load, {0: 10, 1: 10}, CENTERS["maintenance"])
        acc = S.decode(F, L, 10, NR, rng)
        null = S.decode(F, L, 10, NN, rng, scramble=True)
        DEC4E[(regcol, r)] = (acc, null, used)
        print(f"{regcol} {r}: {len(used)} neurons;", {k: round(v, 4) for k, v in summarize(acc, null).items()})
print("variants, hippocampus (atlas), 100 runs and 250 null runs each:")
idx_h = region_idx("hippocampus")
for name, kind, z, pc, need in [("Gaussian, z-scored", "gauss", True, None, {0: 10, 1: 10}),
                                ("boxcar, not normalized", "boxcar", False, None, {0: 10, 1: 10}),
                                ("boxcar, z-scored, 10:20 trials", "boxcar", True, {0: 10, 1: 20}, {0: 10, 1: 20})]:
    F, L, used = build(idx_h, lab_load, need, CENTERS["maintenance"], kind)
    acc = S.decode(F, L, 10, NR, rng, zscore=z, per_class=pc)
    null = S.decode(F, L, 10, NN, rng, scramble=True, zscore=z, per_class=pc)
    print(f"  {name}: {len(used)} neurons;", {k: round(v, 4) for k, v in summarize(acc, null).items()})

**Result: Reproduced.** Hippocampal neurons decode workload during maintenance at a median of 81.4% (paper 80 +/- 8), above the scrambled null (P = 0.004, Bonferroni 0.048, the smallest this run allows); entorhinal (62.5%) and amygdala (59.5%) are not significant after correction. The bundle regions give 81.6% for hippocampus; the variants give 70.0% (Gaussian window), 81.4% (no normalization) and 78.9% (1:2 classes).

### fig4d-encoding-decodes-correct

> During encoding of high-workload trials (set sizes 6 and 8), hippocampal neurons, but not
> entorhinal or amygdala neurons, predicted whether the subject would later respond correctly,
> with a median decoding accuracy of 80%.

Trials of set sizes 6 and 8, incorrect (class 0) against correct (class 1), units from
sessions with at least five incorrect responses among them, five splits (five trials per class
per neuron), steps -4.75 to -3.25 s.

In [ ]:
rng = np.random.default_rng(402)
lab_corr = lambda u: (u["correct"].astype(int), u["set_size"] >= 6)
for regcol in ["region_atlas"]:
    for r in REG:
        F, L, used = build(region_idx(r, regcol), lab_corr, {0: 5, 1: 5}, CENTERS["encoding"])
        acc = S.decode(F, L, 5, NR, rng)
        null = S.decode(F, L, 5, NN, rng, scramble=True)
        print(f"{regcol} {r}: {len(used)} neurons from {UN.sess.iloc[used].nunique()} sessions;",
              {k: round(v, 4) for k, v in summarize(acc, null).items()})
print("variants, hippocampus (atlas):")
for name, kind, z in [("Gaussian, z-scored", "gauss", True), ("boxcar, not normalized", "boxcar", False)]:
    F, L, used = build(region_idx("hippocampus"), lab_corr, {0: 5, 1: 5}, CENTERS["encoding"], kind)
    acc = S.decode(F, L, 5, NR, rng, zscore=z)
    null = S.decode(F, L, 5, NN, rng, scramble=True, zscore=z)
    print(f"  {name}: {len(used)} neurons;", {k: round(v, 4) for k, v in summarize(acc, null).items()})

**Result: Not reproduced.** During encoding of set size 6 and 8 trials, hippocampal neurons (201 from the 10 sessions with at least five errors) decode later correctness at 51.4% (paper 80 +/- 8), at the level of the scrambled null (P = 0.47); no area and no variant is above chance.

### fig4e-workload-decoding-leave-one-subject-out

> Excluding any one subject changed the hippocampal workload decoding accuracy to a value in
> the range 71 to 81%.

The main reading of fig4e (atlas regions, boxcar, z-scored, 10 trials per class), rerun
without the hippocampal units of each subject in turn (100 runs each).

In [ ]:
rng = np.random.default_rng(403)
loso = {}
for sid in sorted(UN.subject.unique()):
    F, L, used = build(region_idx("hippocampus", extra=(UN.subject != sid)), lab_load, {0: 10, 1: 10}, CENTERS["maintenance"])
    acc = S.decode(F, L, 10, NR, rng)
    loso[sid] = (100 * np.median(acc.mean(1)), 100 * np.median(acc), len(used))
L_ = pd.DataFrame(loso, index=["median_a", "median_b", "neurons"]).T
print(L_.round(2).to_string())
print("range, reading (a): %.1f to %.1f%%; reading (b): %.1f to %.1f%%" % (L_.median_a.min(), L_.median_a.max(), L_.median_b.min(), L_.median_b.max()))

**Result: Reproduced.** Leaving out the hippocampal units of one subject gives medians from 71.6% (without subject 1) to 81.4%, inside the widened range 63.9 to 89.1%.

### fig4e-maintenance-neurons-carry-workload

> Workload decoding from hippocampal maintenance neurons alone was little different from
> decoding from all hippocampal neurons, while excluding the maintenance neurons reduced the
> accuracy to 71%.

The decoder of fig4e (main reading) on (i) hippocampal maintenance neurons only and (ii) the
other hippocampal neurons, for the maintenance neurons of each fig2b reading.

In [ ]:
rng = np.random.default_rng(404)
all_h = summarize(*DEC4E[("region_atlas", "hippocampus")][:2])
print("all hippocampal neurons: median (a) %.1f%%, (b) %.1f%%" % (all_h["median_a"], all_h["median_b"]))
for k in ["paired", "unpaired"]:
    for name, extra in [("(i) maintenance neurons only", UN["maint_" + k]), ("(ii) without maintenance neurons", ~UN["maint_" + k])]:
        F, L, used = build(region_idx("hippocampus", extra=extra), lab_load, {0: 10, 1: 10}, CENTERS["maintenance"])
        acc = S.decode(F, L, 10, NR, rng)
        null = S.decode(F, L, 10, NN, rng, scramble=True)
        sm = summarize(acc, null)
        print(f"fig2b {k}, {name}: {len(used)} neurons;", {kk: round(v, 4) for kk, v in sm.items()},
              f"; difference from all, (a) {sm['median_a'] - all_h['median_a']:+.1f} points")

**Result: Reproduced.** The 128 hippocampal maintenance neurons alone decode workload at 81.8%, 0.5 points from all hippocampal neurons (81.4%) and above their null (P = 0.004); the other 512 hippocampal neurons give 71.6% (paper 71 +/- 7.1), below all neurons. The unpaired selection gives 82.3% and 71.6%.

### fig4e-no-decoding-6-vs-8

> The decoder did not distinguish trials of set size 6 from set size 8.

A claim of absence: the decoder of fig4e (main reading), hippocampal neurons, correct trials,
set size 6 (class 0) against 8 (class 1), against its scrambled null at P < 0.05 (not
corrected).

In [ ]:
rng = np.random.default_rng(405)
lab_68 = lambda u: ((u["set_size"] == 8).astype(int), u["correct"] & (u["set_size"] >= 6))
F, L, used = build(region_idx("hippocampus"), lab_68, {0: 10, 1: 10}, CENTERS["maintenance"])
acc = S.decode(F, L, 10, NR, rng)
null = S.decode(F, L, 10, NN, rng, scramble=True)
print(f"hippocampus, set size 6 against 8: {len(used)} neurons;", {k: round(v, 4) for k, v in summarize(acc, null, n_tests=1).items()})

**Result: Reproduced.** Set size 6 against 8 is decoded from hippocampal neurons at 57.7%, not above the scrambled null (P = 0.16).

### fig4f-retrieval-decodes-response

> During retrieval (500 ms before the button press), neurons in all recorded areas predicted
> the IN or OUT response; with all neurons the median decoding accuracy reached 90%.

Correct trials whose press falls within the 2 s stored after probe onset, OUT (class 0)
against IN (class 1), the rate in the 500 ms before the press (one step), 10 splits.

In [ ]:
rng = np.random.default_rng(406)


def build_retrieval(pop_idx):
    F, L, used = [], [], []
    for i in pop_idx:
        u = units[i]
        keep = u["correct"] & (u["rt"] <= 2.0)
        y = u["is_in"].astype(int)
        if np.sum(y[keep] == 0) < 10 or np.sum(y[keep] == 1) < 10:
            continue
        cnt = np.array([np.sum((s >= r - 0.5) & (s < r)) for s, r in zip(u["aligned"], u["rt"])]) / 0.5
        F.append(cnt[keep][:, None])
        L.append(y[keep])
        used.append(i)
    return F, L, used


for regcol in ["region_atlas"]:
    for r in REG + ["all"]:
        F, L, used = build_retrieval(region_idx(r, regcol))
        acc = S.decode(F, L, 10, NR, rng)
        null = S.decode(F, L, 10, NN, rng, scramble=True)
        print(f"{regcol} {r}: {len(used)} neurons;", {k: round(v, 4) for k, v in summarize(acc, null).items()})

**Result: Not reproduced.** In the 500 ms before the press, all neurons decode IN against OUT at 80% (paper 90 +/- 9), with uncorrected P = 0.02 and corrected P = 0.24; no single area is significant (hippocampus 70%, entorhinal 65%, amygdala 65%). Only trials whose press falls within the 2 s stored after the probe can be used.

## Population state space (dPCA)

Trial-averaged rates per set size of all neurons (atlas regions), correct trials, -6 to +2 s
at 20 ms steps, smoothed with a Gaussian kernel over a 1000 ms window (SD 1000/6 ms, the window
as +/- 3 SD, a standard choice; the 20 ms bins replace the Methods' 2 ms bins, which cannot
change a rate smoothed over 1 s). Z-score to fixation: (a) by the mean and SD over trials of
the fixation rate, (b) by the fixation mean and the SD of the rate over the whole trial.
Regularization: (a) the toolbox's cross-validated lambda is not run here (it needs
single-trial pseudopopulations); a fixed lambda of 1e-2 of the data norm is shown instead,
(b) none (pseudo-inverse). The separation of set sizes is compared with 40 dPCA runs on
set-size labels shuffled across each neuron's trials.

### fig4a-dpca-workload-separation

> The first three demixed principal components of the population firing rates explained 44.7%
> of the variance and separated set sizes 4, 6 and 8, with three angles of 120 degrees in the
> dPC2-dPC3 plane; the first 15 components explained 79%.

In [ ]:
rng = np.random.default_rng(501)
TT = np.round(np.arange(-6.0, 2.0, 0.02) + 0.01, 4)
sig_ = (1.0 / 6) / 0.02
kern = np.exp(-0.5 * (np.arange(-25, 26) / sig_) ** 2)
kern /= kern.sum()
pop = [i for i in range(len(units)) if UN.region_atlas[i] in REG]


SM = []     # per neuron: smoothed rates of correct trials, their set sizes, fixation mean and SDs
for i in pop:
    u = units[i]
    c = u["correct"]
    r = np.apply_along_axis(lambda x: np.convolve(x, kern, mode="same"), 1, u["bins"][c].astype(float)) / 0.02
    fix_tr = win_rate(u, S.FIX)[c]
    SM.append((r, u["set_size"][c], fix_tr.mean(), fix_tr.std(ddof=1), r.mean(0).std()))


def psth_z(k, mode, ss=None):
    r, s, m, sd_a, sd_b = SM[k]
    s = s if ss is None else ss
    sd = sd_a if mode == "a" else sd_b
    return (np.array([r[s == v].mean(0) for v in (4, 6, 8)]) - m) / (sd if sd > 0 else 1.0)


XZ = {mode: np.array([psth_z(k, mode) for k in range(len(pop))]) for mode in ("a", "b")}
maint_t = (TT >= -3) & (TT < 0)


def angles(comps):
    P = np.array([[comps[1]["z"][s][maint_t].mean(), comps[2]["z"][s][maint_t].mean()] for s in range(3)])
    v = P - P.mean(0)
    ang = np.degrees(np.arctan2(v[:, 1], v[:, 0]))
    order = np.argsort(ang)
    a = np.diff(np.r_[ang[order], ang[order][0] + 360])
    return np.sort(a), P


DP = {}
for mode in ("a", "b"):
    for lam in (0.0, 1e-2):
        comps, Xf, tot = S.dpca(XZ[mode], 15, lam)
        DP[(mode, lam)] = comps
        ev3, ev15 = S.cumulative_ev(comps, Xf, tot, 3), S.cumulative_ev(comps, Xf, tot, 15)
        ang, P = angles(comps)
        print(f"z-score ({mode}), lambda {lam}: first 3 dPCs {ev3:.1f}% (sum of single {sum(c['ev'] for c in comps[:3]):.1f}%), "
              f"first 15 {ev15:.1f}%; marginalizations of dPC1-3 {[c['marg'] for c in comps[:3]]}; "
              f"angles in the dPC2-dPC3 plane {np.round(ang, 1).tolist()}")
# separation against label-shuffled dPCA (main: z-score a, no regularization)
def pair_dists(comps):
    Z = np.array([[c["z"][s][maint_t] for c in comps[:15]] for s in range(3)])   # set x comp x time
    return [float(np.mean(np.sqrt(((Z[i] - Z[j]) ** 2).sum(0)))) for i, j in [(0, 1), (0, 2), (1, 2)]]


obs_d = pair_dists(DP[("a", 0.0)])
null_d = []
for _ in range(40):
    Xs = [psth_z(k, "a", rng.permutation(SM[k][1])) for k in range(len(pop))]
    null_d.append(pair_dists(S.dpca(np.array(Xs), 15, 0.0)[0]))
null_d = np.array(null_d)
print("maintenance distance between set sizes (4-6, 4-8, 6-8):", np.round(obs_d, 3).tolist(),
      "; label-shuffled 95th percentiles:", np.round(np.percentile(null_d, 95, axis=0), 3).tolist(),
      "; P:", [(np.sum(null_d[:, k] >= obs_d[k]) + 1) / 41 for k in range(3)])

**Result: Not reproduced.** The first three dPCs explain 32.2% (paper 44.7 +/- 4.47) and the first 15 explain 64.3% (79 +/- 7.9) with the fixation-SD z-score; 18.7% and 52.3% with the other z-score. dPC2 is a time component, so the three set sizes lie on a line in the dPC2-dPC3 plane (angles 0, 180, 180 degrees, not 120). In the first 15 dPCs, set size 4 is farther from 6 and from 8 during maintenance than in 40 label-shuffled runs (P = 0.024), but 6 and 8 are not (P = 0.68), so the partial criterion is not met either.

### fig4b-speed-lowest-maintenance

> In the dPCA state space, the speed of the population trajectory was highest during encoding
> and lowest during maintenance, lower than in each other trial period, including fixation.

First 15 dPCs, speed by Methods equation 2 (step 20 ms), averaged over set sizes and over each
period; dPCA rerun on neurons subsampled at 20% with replacement, (i) 200 runs, (ii) 2000
runs; z-score (a), no regularization. Retrieval (a) 0 to 1 s, (b) 0 to 2 s. Permutation t test
(2000 permutations) of the per-run maintenance speeds against each other period.

### fig4c-distance-largest-maintenance

> The pairwise distance between the trajectories of different set sizes was larger during
> maintenance than during encoding, and the largest of the trial periods.

Distance by Methods equation 1 between each pair of set-size trajectories in the first 15
dPCs, averaged over pairs and periods, from the same runs.

In [ ]:
rng = np.random.default_rng(502)
PER = {"fixation": (-6, -5), "encoding": (-5, -3), "maintenance": (-3, 0), "retrieval_a": (0, 1), "retrieval_b": (0, 2)}


def speed_dist(comps):
    Z = np.array([[c["z"][s] for c in comps[:15]] for s in range(3)])     # set x comp x time
    V = np.sqrt((np.diff(Z, axis=2) ** 2).sum(1)) / 15 / 0.02            # set x (time - 1)
    Dd = np.mean([np.sqrt(((Z[i] - Z[j]) ** 2).sum(0)) for i, j in [(0, 1), (0, 2), (1, 2)]], axis=0)
    out = {}
    for k, (a, b) in PER.items():
        m = (TT >= a) & (TT < b)
        out["V_" + k] = V.mean(0)[m[1:]].mean()
        out["D_" + k] = Dd[m].mean()
    return out


runs_sd = []
for r_ in range(2000):
    sub = rng.integers(0, len(pop), int(round(0.2 * len(pop))))
    runs_sd.append(speed_dist(S.dpca(XZ["a"][sub], 15, 0.0)[0]))
RS = pd.DataFrame(runs_sd)
for n_runs in (200, 2000):
    Rk = RS.iloc[:n_runs]
    print(f"== {n_runs} runs; per-period means (speed in z per s, distance in z):")
    print(Rk.mean().round(4).to_string())
    for kind in ("V", "D"):
        for ret in ("retrieval_a", "retrieval_b"):
            others = ["fixation", "encoding", ret]
            res = []
            for o in others:
                t, p = S.perm_t(Rk[f"{kind}_maintenance"], Rk[f"{kind}_{o}"], n_perm=2000, rng=rng)
                res.append(f"{o}: t = {t:.2f}, P = {p:.4f}")
            vals = {o: Rk[f"{kind}_{o}"].mean() for o in ["maintenance"] + others}
            print(f"   {'speed' if kind == 'V' else 'distance'}, {ret}: maintenance against " + "; ".join(res)
                  + f"; highest period {max(vals, key=vals.get)}, lowest {min(vals, key=vals.get)}")
full = speed_dist(DP[("a", 0.0)])
print("full population (no subsampling):", {k: round(v, 4) for k, v in full.items()})

**Result: Reproduced with 2000 runs: the distance between set-size trajectories is largest during maintenance (7.**67) and larger than during encoding (7.49), t = 4.96, P = 0.0005. With 200 runs maintenance is still the largest, but its excess over encoding is not significant (7.52 against 7.48, P = 0.74), which is partial under the criterion.

**Result: Partially reproduced.** Speed is lowest during maintenance (0.74 z/s over 200 runs, below fixation 1.07, encoding 0.95 and retrieval 1.41 or 1.29; P = 0.0005 for each), but it is highest during retrieval, not encoding, under both retrieval windows and both run counts.

### figs3d-hippocampus-loadings

> Ranking the first 200 neurons of each region by their dPCA loading, hippocampal neurons had
> higher loadings than entorhinal and amygdala neurons at each rank except the first three,
> where entorhinal neurons had the highest loading.

The main dPCA (z-score a, no regularization). Component (a) dPC2, (b) dPC3; loading (i) the
absolute decoder weight, (ii) the signed weight. Per region the 200 largest, sorted; paired
Wilcoxon signed-rank test of hippocampus against entorhinal cortex over the ranks.

In [ ]:
regs_pop = UN.region_atlas.to_numpy()[pop]
for comp_name, ci in [("(a) dPC2", 1), ("(b) dPC3", 2)]:
    w = DP[("a", 0.0)][ci]["decoder"]
    for lname, f_ in [("(i) absolute", np.abs), ("(ii) signed", lambda x: x)]:
        top = {r: np.sort(f_(w[regs_pop == r]))[::-1][:200] for r in REG}
        wt = stats.wilcoxon(top["hippocampus"], top["entorhinal"])
        print(f"{comp_name} [{DP[('a', 0.0)][ci]['marg']}], {lname}: hippocampus above entorhinal at {int(np.sum(top['hippocampus'] > top['entorhinal']))} of 200 ranks "
              f"(Wilcoxon P = {wt.pvalue:.2g}), above amygdala at {int(np.sum(top['hippocampus'] > top['amygdala']))}; "
              f"ranks 1-3 hipp {np.round(top['hippocampus'][:3], 4).tolist()} ent {np.round(top['entorhinal'][:3], 4).tolist()}")

**Result: Reproduced.** Under every component and loading reading, the hippocampal top-200 loadings exceed the entorhinal and the amygdalar ones at all 200 ranks (Wilcoxon P = 1.4e-34 against entorhinal cortex). The exception the statement mentions, entorhinal neurons highest at the first three ranks, does not appear in any reading; the criterion does not judge it.

## Claims tested in the second notebook

### sfc-maintenance-over-encoding

> Maintenance neurons showed significant spike-field coherence to the local field potential, but not to the iEEG or scalp EEG, and for all frequency bins the percentage of neurons with spike-field coherence was higher during maintenance than during encoding.

The claim needs the microwire LFP (`ecephys.lfp`, 32 kHz) of every wire that carries a hippocampal maintenance neuron with at least 100 spikes in maintenance and in encoding. The cell below counts those wires and estimates the streaming time from the rate measured when the EEG and iEEG were read above. Added to this notebook it would bring it close to the 30 minute limit, so the claim is tested in `reproduce_boran_2019_sfc.ipynb`.

In [ ]:
rows = []
for k in ["paired", "unpaired"]:
    idx = [i for i in np.where((UN.region_atlas == "hippocampus") & UN["maint_" + k])[0]
           if win_counts(units[i], S.ENC)[units[i]["correct"]].sum() >= 100 and win_counts(units[i], S.MAINT)[units[i]["correct"]].sum() >= 100]
    wires = {(UN.sess[i], [u for u in D if u["subject"] + "-" + u["session"] == UN.sess[i]][0]["units"][UN.unit[i]]["electrode_row"]) for i in idx}
    mb = sum(SIG[[d["path"] for d in D if d["subject"] + "-" + d["session"] == s][0]]["ieeg"].shape[0] * 8 * 32000 * 8 for s, _ in wires) / 1e6
    rows.append({"selection": k, "neurons": len(idx), "wires": len(wires), "LFP_MB": round(mb), "est_minutes": round(mb / (mb_sig / t_sig) / 60, 1)})
print(pd.DataFrame(rows).to_string(index=False))
print("expected neurons (fig. S6b): 94")

The outcome is in the second notebook. 

### hfo-rates-unaffected (blocked, methods underspecified)

> The rates of clinically relevant high-frequency oscillations were not affected by performing the task.

The claims file blocks this claim until the HFO detector of reference 27 (Fedele et al. 2017) is supplied; neither the detector nor the comparison is given in the paper. The files hold the iEEG at 2 kHz, which an HFO detector would need, but only during the 8 s of each trial.

## Summary

| Claim | Outcome |
|---|---|
| behavior-accuracy-set-size | Reproduced |
| behavior-capacity | Reproduced under reading (c), rates pooled over set sizes times 8: mean K = 6 |
| behavior-rt-workload | Reproduced |
| behavior-rt-correct-incorrect | Reproduced |
| neurons-recorded | Not reproduced |
| spike-sorting-quality | Partially reproduced |
| fig2b-maintenance-neurons-hippocampus | Reproduced |
| fig2c-load-dependent-maintenance-neurons | Reproduced under the criterion |
| fig3e-probe-neurons-entorhinal | Reproduced under the criterion with window (b), the probe to the press, and the unpaired test: entorhinal cortex has the highest percentage (45 of 421, 10 |
| fig3-probe-onset-not-movement | Reproduced under one reading |
| fig3-probe-neurons-few-task-specific | Partially reproduced |
| fig4a-dpca-workload-separation | Not reproduced |
| fig4b-speed-lowest-maintenance | Partially reproduced |
| fig4c-distance-largest-maintenance | Reproduced with 2000 runs: the distance between set-size trajectories is largest during maintenance (7 |
| fig4d-encoding-decodes-correct | Not reproduced |
| fig4e-maintenance-decodes-workload | Reproduced |
| fig4e-workload-decoding-leave-one-subject-out | Reproduced |
| fig4e-maintenance-neurons-carry-workload | Reproduced |
| fig4e-no-decoding-6-vs-8 | Reproduced |
| fig4f-retrieval-decodes-response | Not reproduced |
| figs3d-hippocampus-loadings | Reproduced |
| fig5-subject1-eeg-power-workload | Partially reproduced |
| fig5-subject1-hippocampal-power-workload | Reproduced |
| fig5-subject1-neurons-workload | Partially reproduced |
| fig5-subject1-plv-hippocampus-pz | Partially reproduced |
| fig6-plv-all-subjects | Partially reproduced |
| tables1-plv-per-subject | Not reproduced |
| fig6-plv-workload-eight-of-nine | Partially reproduced |
| fig6-plv-maintenance-over-encoding | Not reproduced |
| fig6-power-unchanged-individual-band | Not reproduced, resting on an assumption |
| fig6i-plv-posterior-gradient | Not reproduced |
| fig6i-no-left-right-difference | Reproduced |
| soz-plv-no-difference | Reproduced |
| hfo-rates-unaffected | Blocked: methods underspecified |
| sfc-maintenance-over-encoding | Partially reproduced (second notebook) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's:

- 37 sessions, where the paper has 36; 1487 units, where it has 1526. Eight sessions hold no
  units table.
- Regions of units: the atlas label of each bundle's deepest contact (Methods, Depth electrode
  localization) for every claim except `neurons-recorded`, where the claims file names the
  bundle; the bundle reading is shown for the main neuron and decoding claims.
- In the null repetitions of the probe-neuron selection, the P of each t test comes from the t
  distribution (agreement with the permutation P on the observed tests printed).
- Decoding uses 100 resampling runs and 250 null runs (Methods: 500 and 500); the NDT details
  the Methods leave open take the toolbox's defaults.
- The files store only 8 s per trial (-6 to +2 s around the probe), which limits the
  retrieval and button-aligned analyses to presses within 2 s.
- dPCA uses 20 ms bins, no cross-validated regularization, and 50 label-shuffled runs.
- The scalp EEG is re-referenced to the averaged mastoids here; the deposited EEG is not.
- PLV follows FieldTrip's taper-averaged cross-spectra (assumption); per-taper PLV is a
  variant for the selection.
- Spike-field coherence is tested in the second notebook, with a window around each spike the
  paper does not give; the HFO detector is not available.

What was found about the dataset:

- 1487 units in 29 of 37 sessions; the per-session mean over sessions with units (51.3) matches
  the paper's 51, which suggests the paper's 1526 includes about one more session of units.
- Units point at macroelectrode rows of the electrodes table, and the microwire LFP channels
  reuse those rows; microwires have no rows or coordinates of their own.
- Region counts by bundle name differ widely from the paper's; the atlas label of the bundle's
  deepest contact comes close (640, 421, 398 against 676, 449, 401).
- Spike times are stored unsorted in some units.
- `response_type` uses 51/52 on correct and 1/2 on incorrect trials, without a description.
- The trial-stage table numbers each Probe stage one trial late, and seven files lack it.
- EEG and iEEG are labelled volts but hold values of about +/- 150 and +/- 240, consistent with
  microvolts; the electrodes table says the EEG is referenced to the averaged mastoids, but A1
  + A2 is not zero.
- Electrode coordinates use a frame with +x posterior, not MNI; scalp electrode positions are
  template positions on a sphere.
- In subject 2 session 5 the microwire LFP is shorter than 32 kHz x 8 s per trial (second
  notebook).
- The subject 1 description lists AHR among SOZ electrodes, but subject 1 has no AHR electrode.